# W02_holevo: Holevo-CIL giai đoạn A (sàng lọc), ImageNet-R seed 1993 (Kaggle RTX PRO 6000)

Hướng A: adapter được huấn luyện thêm −λ·χ̃, với χ là Holevo χ của mọi lớp đã thấy (lớp cũ: trạng thái dựng từ mean + cov đã lưu; lớp mới: queue feature của task hiện tại). Spec: `exps/holevo/README.md`.

3 run song song trên ImageNet-R seed 1993 (run sàng lọc): `holevo_vn` (phương pháp chính), `holevo_mcr2` (bản classical tương đương: log-det MCR²), `holevo_spread` (chỉ S(ρ̄)). λ tự hiệu chỉnh ở epoch 0 của task 1 theo tỉ số gradient; chẩn đoán χ_oracle bật (đọc ảnh lớp cũ, chỉ để phân tích). Ước tính 2,5–3 giờ.

`SUMMARY.txt` tính các luật R/G/M/O/P so với baseline 4 run `noise` (đã đăng ký trong `scripts/summarize_holevo.py`). Held-out sẽ ghi "chưa đủ run (0/5)".

**Code:** base `c7bc4fd` (dataset `qgrsa-base-c7bc4fd`, tạo bằng `python scripts/pack_base.py --commit c7bc4fd`) + patch 20 file nhúng trong cell 2 (build 2026-10-30 20:41):
- `.gitignore`
- `exps/holevo/holevo_mcr2.json`
- `exps/holevo/holevo_spread.json`
- `exps/holevo/holevo_vn.json`
- `exps/holevo_povm/hp_diag.json`
- `exps/holevo_povm/hp_full.json`
- `exps/holevo_povm/hp_povm.json`
- `exps/holevo_povm/hp_twin.json`
- `exps/holevo_povm/hp_twin_povm.json`
- `exps/svf/svf.json`
- `models/RSIAT_adapter.py`
- `scripts/build_experiments_notebook.py`
- `scripts/build_wave.py`
- `scripts/pack_base.py`
- `scripts/summarize_holevo.py`
- `scripts/summarize_svf.py`
- `tests/test_holevo.py`
- `tests/test_svf.py`
- `utils/holevo.py`
- `utils/qsd.py`

**Trước khi chạy:** GPU **RTX PRO 6000**; *Add Input* > `qgrsa-base-c7bc4fd`, `zaphat206/imagenet-r`, `ViT-B/16 224 in21k`; Internet có thể Off. Rồi *Save Version > Save & Run All (Commit)*.

**Gửi lại:** `results_W02_holevo.zip` (hoặc `runs/SUMMARY.txt`).

In [ ]:
%%writefile /kaggle/working/run.py
"""W02_holevo (base c7bc4fd). Sửa PLAN ở đây để đổi job; đổi code thì build lại wave."""
PLAN = {'name': 'W02_holevo',
 'jobs': [[['imagenetr'], [1993], ['holevo_vn', 'holevo_mcr2', 'holevo_spread']]],
 'max_parallel': 3,
 'overrides': {'save_checkpoints': False},
 'tests': ['test_holevo.py', 'test_qsd.py'],
 'summaries': ['scripts/summarize_holevo.py']}
PATCH = {'base': 'c7bc4fd4ddcae36d3023e1aa9935fd97e9c797a0', 'built': '2026-10-30 20:41', 'files': {'.gitignore': 'IyBQeXRob24gY2FjaGVzIGFuZCB0ZXN0IHN0YXRlCl9fcHljYWNoZV9fLwoqLnB5W2NvZF0KLnB5dGVzdF9jYWNoZS8KLmNvdmVyYWdlCmh0bWxjb3YvCgojIExvY2FsIGVudmlyb25tZW50cyBhbmQgZWRpdG9yIGZpbGVzCi52ZW52Lwp2ZW52LwouaWRlYS8KLnZzY29kZS8KLkRTX1N0b3JlClRodW1icy5kYgoKIyBOb3RlYm9vay1nZW5lcmF0ZWQgc3RhdGUKLmlweW5iX2NoZWNrcG9pbnRzLwoKIyBEYXRhc2V0cyBhbmQgZG93bmxvYWRlZCBtb2RlbCBhc3NldHMKZGF0YXNldHMvCmRhdGEvZGF0YXNldHMvCioucHQKKi5wdGgKKi5wa2wKKi5zYWZldGVuc29ycwoqLmJpbgoKIyBSdW50aW1lIG91dHB1dHMKY2twdC8KbG9ncy8Kb3V0cHV0cy8KcnVucy8KYXJ0aWZhY3RzLwpjYWNoZS8KCiMgUmVidWlsZGFibGUgcGFja2FnZXMgYW5kIGdlbmVyYXRlZCByZXBvcnRzCmRpc3QvCnJlcG9ydHMvZ2VuZXJhdGVkLwpyZXBvcnRzL3FicmlkZ2VfdjFfMS8KCiMgVGVtcG9yYXJ5IGZpbGVzCioudG1wCioudGVtcAoqLmxvZy50bXAKCiMgRXhwZXJpbWVudCByZXN1bHRzIChleHBzLzxleHBlcmltZW50Pi9yZXN1bHRzLy4uLik6IGtlZXAgcGVyLXJ1biAubG9nLCBjb25maWdzLAojIHN1bW1hcmllcywgUkVBRE1FcyBhbmQgYW5hbHlzaXMgc2NyaXB0cyBpbiBnaXQuIFBlci10YXNrIHN0YXRzIChucHovanNvbi9zY29yZXMpCiMgYW5kIGNvbnNvbGUgZHVtcHMgc3RheSBsb2NhbDogdGhleSBhcmUgbGFyZ2UgYW5kIG9ubHkgbmVlZGVkIGZvciBsb2NhbCBhbmFseXNlcy4KIWV4cHMvKiovbG9ncy8KZXhwcy8qKi8qX3N0YXRzLwpleHBzLyoqL2NvbnNvbGUvCgojIER1cGxpY2F0ZXMgd2FpdGluZyBmb3IgbWFudWFsIGRlbGV0aW9uCl9kdXBsaWNhdGVzX3RvX2RlbGV0ZS8Kd2F2ZXMvdzAxX3N2Zi9yZXN1bHRzLw==', 'exps/holevo/holevo_mcr2.json': 'ewogICJwcmVmaXgiOiAiaG9sZXZvX21jcjIiLAogICJzc2NhX21vZGUiOiAibGVnYWN5IiwKICAicm5nX3JlcGxpY2F0ZSI6IDAsCiAgInN0YXRzX2RpYWdub3N0aWNzIjogZmFsc2UsCiAgInNhdmVfcHJlZGljdGlvbnMiOiB0cnVlLAogICJzYXZlX2NoZWNrcG9pbnRzIjogZmFsc2UsCiAgIm9mZmxpbmUiOiB0cnVlLAogICJob2xldm9fbW9kZSI6ICJtY3IyIiwKICAiaG9sZXZvX2xhbWJkYSI6ICJhdXRvIiwKICAiaG9sZXZvX2dyYWRfcmF0aW8iOiAwLjUsCiAgImhvbGV2b19jYWxpYl9zdGVwcyI6IDIwLAogICJob2xldm9fcXVldWVfY2FwIjogMjU2LAogICJob2xldm9fZXBzIjogMWUtMDgsCiAgImhvbGV2b19tY3IyX2VwczIiOiAwLjUsCiAgImhvbGV2b19vcmFjbGVfZGlhZyI6IHRydWUsCiAgImhvbGV2b19sb2dfZXZlcnkiOiAyNQp9Cg==', 'exps/holevo/holevo_spread.json': 'ewogICJwcmVmaXgiOiAiaG9sZXZvX3NwcmVhZCIsCiAgInNzY2FfbW9kZSI6ICJsZWdhY3kiLAogICJybmdfcmVwbGljYXRlIjogMCwKICAic3RhdHNfZGlhZ25vc3RpY3MiOiBmYWxzZSwKICAic2F2ZV9wcmVkaWN0aW9ucyI6IHRydWUsCiAgInNhdmVfY2hlY2twb2ludHMiOiBmYWxzZSwKICAib2ZmbGluZSI6IHRydWUsCiAgImhvbGV2b19tb2RlIjogInNwcmVhZCIsCiAgImhvbGV2b19sYW1iZGEiOiAiYXV0byIsCiAgImhvbGV2b19ncmFkX3JhdGlvIjogMC41LAogICJob2xldm9fY2FsaWJfc3RlcHMiOiAyMCwKICAiaG9sZXZvX3F1ZXVlX2NhcCI6IDI1NiwKICAiaG9sZXZvX2VwcyI6IDFlLTA4LAogICJob2xldm9fbWNyMl9lcHMyIjogMC41LAogICJob2xldm9fb3JhY2xlX2RpYWciOiB0cnVlLAogICJob2xldm9fbG9nX2V2ZXJ5IjogMjUKfQo=', 'exps/holevo/holevo_vn.json': 'ewogICJwcmVmaXgiOiAiaG9sZXZvX3ZuIiwKICAic3NjYV9tb2RlIjogImxlZ2FjeSIsCiAgInJuZ19yZXBsaWNhdGUiOiAwLAogICJzdGF0c19kaWFnbm9zdGljcyI6IGZhbHNlLAogICJzYXZlX3ByZWRpY3Rpb25zIjogdHJ1ZSwKICAic2F2ZV9jaGVja3BvaW50cyI6IGZhbHNlLAogICJvZmZsaW5lIjogdHJ1ZSwKICAiaG9sZXZvX21vZGUiOiAidm4iLAogICJob2xldm9fbGFtYmRhIjogImF1dG8iLAogICJob2xldm9fZ3JhZF9yYXRpbyI6IDAuNSwKICAiaG9sZXZvX2NhbGliX3N0ZXBzIjogMjAsCiAgImhvbGV2b19xdWV1ZV9jYXAiOiAyNTYsCiAgImhvbGV2b19lcHMiOiAxZS0wOCwKICAiaG9sZXZvX21jcjJfZXBzMiI6IDAuNSwKICAiaG9sZXZvX29yYWNsZV9kaWFnIjogdHJ1ZSwKICAiaG9sZXZvX2xvZ19ldmVyeSI6IDI1Cn0K', 'exps/holevo_povm/hp_diag.json': 'ewogICJwcmVmaXgiOiAiaHBfZGlhZyIsCiAgInNzY2FfbW9kZSI6ICJsZWdhY3kiLAogICJybmdfcmVwbGljYXRlIjogMCwKICAic3RhdHNfZGlhZ25vc3RpY3MiOiBmYWxzZSwKICAic2F2ZV9wcmVkaWN0aW9ucyI6IHRydWUsCiAgInNhdmVfY2hlY2twb2ludHMiOiBmYWxzZSwKICAib2ZmbGluZSI6IHRydWUsCiAgImhwX21vZGUiOiAiZGlhZyIsCiAgImhvbGV2b19sYW1iZGEiOiAiYXV0byIsCiAgImhvbGV2b19ncmFkX3JhdGlvIjogMC41LAogICJob2xldm9fY2FsaWJfc3RlcHMiOiAyMCwKICAiaG9sZXZvX3F1ZXVlX2NhcCI6IDI1NiwKICAiaG9sZXZvX2VwcyI6IDFlLTA4LAogICJob2xldm9fbWNyMl9lcHMyIjogMC41LAogICJob2xldm9fb3JhY2xlX2RpYWciOiB0cnVlLAogICJob2xldm9fbG9nX2V2ZXJ5IjogMjUsCiAgImhwX3RhdSI6IDEuMCwKICAiaHBfZXBzX29tZWdhIjogMWUtMDYsCiAgImhwX2dyYWRfcmF0aW8iOiAwLjUsCiAgInFzZF9oZWFkcyI6IFsKICAgICJsaW5lYXIiLAogICAgInJpZGdlX3N0YXRzIiwKICAgICJwZ21fc3RhdHMiCiAgXSwKICAicXNkX2RlbHRhIjogMS4zMzE2LAogICJxc2RfcmlkZ2UiOiAwLjEsCiAgInFzZF9zZWVkIjogMjAyNjEwMjIsCiAgInFzZF9iYXRjaCI6IDEyOCwKICAicXNkX2Z1c2UiOiB0cnVlLAogICJxc2Rfc2F2ZV9zY29yZXMiOiBmYWxzZQp9Cg==', 'exps/holevo_povm/hp_full.json': 'ewogICJwcmVmaXgiOiAiaHBfZnVsbCIsCiAgInNzY2FfbW9kZSI6ICJsZWdhY3kiLAogICJybmdfcmVwbGljYXRlIjogMCwKICAic3RhdHNfZGlhZ25vc3RpY3MiOiBmYWxzZSwKICAic2F2ZV9wcmVkaWN0aW9ucyI6IHRydWUsCiAgInNhdmVfY2hlY2twb2ludHMiOiBmYWxzZSwKICAib2ZmbGluZSI6IHRydWUsCiAgImhwX21vZGUiOiAiZnVsbCIsCiAgImhvbGV2b19sYW1iZGEiOiAiYXV0byIsCiAgImhvbGV2b19ncmFkX3JhdGlvIjogMC41LAogICJob2xldm9fY2FsaWJfc3RlcHMiOiAyMCwKICAiaG9sZXZvX3F1ZXVlX2NhcCI6IDI1NiwKICAiaG9sZXZvX2VwcyI6IDFlLTA4LAogICJob2xldm9fbWNyMl9lcHMyIjogMC41LAogICJob2xldm9fb3JhY2xlX2RpYWciOiB0cnVlLAogICJob2xldm9fbG9nX2V2ZXJ5IjogMjUsCiAgImhwX3RhdSI6IDEuMCwKICAiaHBfZXBzX29tZWdhIjogMWUtMDYsCiAgImhwX2dyYWRfcmF0aW8iOiAwLjUsCiAgInFzZF9oZWFkcyI6IFsKICAgICJsaW5lYXIiLAogICAgInJpZGdlX3N0YXRzIiwKICAgICJwZ21fc3RhdHMiCiAgXSwKICAicXNkX2RlbHRhIjogMS4zMzE2LAogICJxc2RfcmlkZ2UiOiAwLjEsCiAgInFzZF9zZWVkIjogMjAyNjEwMjIsCiAgInFzZF9iYXRjaCI6IDEyOCwKICAicXNkX2Z1c2UiOiB0cnVlLAogICJxc2Rfc2F2ZV9zY29yZXMiOiBmYWxzZQp9Cg==', 'exps/holevo_povm/hp_povm.json': 'ewogICJwcmVmaXgiOiAiaHBfcG92bSIsCiAgInNzY2FfbW9kZSI6ICJsZWdhY3kiLAogICJybmdfcmVwbGljYXRlIjogMCwKICAic3RhdHNfZGlhZ25vc3RpY3MiOiBmYWxzZSwKICAic2F2ZV9wcmVkaWN0aW9ucyI6IHRydWUsCiAgInNhdmVfY2hlY2twb2ludHMiOiBmYWxzZSwKICAib2ZmbGluZSI6IHRydWUsCiAgImhwX21vZGUiOiAicG92bSIsCiAgImhvbGV2b19sYW1iZGEiOiAiYXV0byIsCiAgImhvbGV2b19ncmFkX3JhdGlvIjogMC41LAogICJob2xldm9fY2FsaWJfc3RlcHMiOiAyMCwKICAiaG9sZXZvX3F1ZXVlX2NhcCI6IDI1NiwKICAiaG9sZXZvX2VwcyI6IDFlLTA4LAogICJob2xldm9fbWNyMl9lcHMyIjogMC41LAogICJob2xldm9fb3JhY2xlX2RpYWciOiB0cnVlLAogICJob2xldm9fbG9nX2V2ZXJ5IjogMjUsCiAgImhwX3RhdSI6IDEuMCwKICAiaHBfZXBzX29tZWdhIjogMWUtMDYsCiAgImhwX2dyYWRfcmF0aW8iOiAwLjUsCiAgInFzZF9oZWFkcyI6IFsKICAgICJsaW5lYXIiLAogICAgInJpZGdlX3N0YXRzIiwKICAgICJwZ21fc3RhdHMiCiAgXSwKICAicXNkX2RlbHRhIjogMS4zMzE2LAogICJxc2RfcmlkZ2UiOiAwLjEsCiAgInFzZF9zZWVkIjogMjAyNjEwMjIsCiAgInFzZF9iYXRjaCI6IDEyOCwKICAicXNkX2Z1c2UiOiB0cnVlLAogICJxc2Rfc2F2ZV9zY29yZXMiOiBmYWxzZQp9Cg==', 'exps/holevo_povm/hp_twin.json': 'ewogICJwcmVmaXgiOiAiaHBfdHdpbiIsCiAgInNzY2FfbW9kZSI6ICJsZWdhY3kiLAogICJybmdfcmVwbGljYXRlIjogMCwKICAic3RhdHNfZGlhZ25vc3RpY3MiOiBmYWxzZSwKICAic2F2ZV9wcmVkaWN0aW9ucyI6IHRydWUsCiAgInNhdmVfY2hlY2twb2ludHMiOiBmYWxzZSwKICAib2ZmbGluZSI6IHRydWUsCiAgImhwX21vZGUiOiAidHdpbiIsCiAgImhvbGV2b19sYW1iZGEiOiAiYXV0byIsCiAgImhvbGV2b19ncmFkX3JhdGlvIjogMC41LAogICJob2xldm9fY2FsaWJfc3RlcHMiOiAyMCwKICAiaG9sZXZvX3F1ZXVlX2NhcCI6IDI1NiwKICAiaG9sZXZvX2VwcyI6IDFlLTA4LAogICJob2xldm9fbWNyMl9lcHMyIjogMC41LAogICJob2xldm9fb3JhY2xlX2RpYWciOiB0cnVlLAogICJob2xldm9fbG9nX2V2ZXJ5IjogMjUsCiAgImhwX3RhdSI6IDEuMCwKICAiaHBfZXBzX29tZWdhIjogMWUtMDYsCiAgImhwX2dyYWRfcmF0aW8iOiAwLjUsCiAgInFzZF9oZWFkcyI6IFsKICAgICJsaW5lYXIiLAogICAgInJpZGdlX3N0YXRzIiwKICAgICJwZ21fc3RhdHMiCiAgXSwKICAicXNkX2RlbHRhIjogMS4zMzE2LAogICJxc2RfcmlkZ2UiOiAwLjEsCiAgInFzZF9zZWVkIjogMjAyNjEwMjIsCiAgInFzZF9iYXRjaCI6IDEyOCwKICAicXNkX2Z1c2UiOiB0cnVlLAogICJxc2Rfc2F2ZV9zY29yZXMiOiBmYWxzZQp9Cg==', 'exps/holevo_povm/hp_twin_povm.json': 'ewogICJwcmVmaXgiOiAiaHBfdHdpbl9wb3ZtIiwKICAic3NjYV9tb2RlIjogImxlZ2FjeSIsCiAgInJuZ19yZXBsaWNhdGUiOiAwLAogICJzdGF0c19kaWFnbm9zdGljcyI6IGZhbHNlLAogICJzYXZlX3ByZWRpY3Rpb25zIjogdHJ1ZSwKICAic2F2ZV9jaGVja3BvaW50cyI6IGZhbHNlLAogICJvZmZsaW5lIjogdHJ1ZSwKICAiaHBfbW9kZSI6ICJ0d2luX3Bvdm0iLAogICJob2xldm9fbGFtYmRhIjogImF1dG8iLAogICJob2xldm9fZ3JhZF9yYXRpbyI6IDAuNSwKICAiaG9sZXZvX2NhbGliX3N0ZXBzIjogMjAsCiAgImhvbGV2b19xdWV1ZV9jYXAiOiAyNTYsCiAgImhvbGV2b19lcHMiOiAxZS0wOCwKICAiaG9sZXZvX21jcjJfZXBzMiI6IDAuNSwKICAiaG9sZXZvX29yYWNsZV9kaWFnIjogdHJ1ZSwKICAiaG9sZXZvX2xvZ19ldmVyeSI6IDI1LAogICJocF90YXUiOiAxLjAsCiAgImhwX2Vwc19vbWVnYSI6IDFlLTA2LAogICJocF9ncmFkX3JhdGlvIjogMC41LAogICJxc2RfaGVhZHMiOiBbCiAgICAibGluZWFyIiwKICAgICJyaWRnZV9zdGF0cyIsCiAgICAicGdtX3N0YXRzIgogIF0sCiAgInFzZF9kZWx0YSI6IDEuMzMxNiwKICAicXNkX3JpZGdlIjogMC4xLAogICJxc2Rfc2VlZCI6IDIwMjYxMDIyLAogICJxc2RfYmF0Y2giOiAxMjgsCiAgInFzZF9mdXNlIjogdHJ1ZSwKICAicXNkX3NhdmVfc2NvcmVzIjogZmFsc2UKfQo=', 'exps/svf/svf.json': 'ewogICJwcmVmaXgiOiAic3ZmIiwKICAic3NjYV9tb2RlIjogImxlZ2FjeSIsCiAgInJuZ19yZXBsaWNhdGUiOiAwLAogICJzdGF0c19kaWFnbm9zdGljcyI6IGZhbHNlLAogICJzYXZlX3ByZWRpY3Rpb25zIjogdHJ1ZSwKICAic2F2ZV9jaGVja3BvaW50cyI6IGZhbHNlLAogICJxc2RfaGVhZHMiOiBbCiAgICAibGluZWFyIiwKICAgICJyaWRnZV9zdGF0cyIsCiAgICAic25hcF9yaWRnZSIKICBdLAogICJxc2RfcmlkZ2UiOiAwLjEsCiAgInFzZF9zZWVkIjogMjAyNjEwMjIsCiAgInFzZF9iYXRjaCI6IDEyOCwKICAicXNkX2Z1c2UiOiB0cnVlLAogICJxc2Rfc2F2ZV9zY29yZXMiOiB0cnVlLAogICJvZmZsaW5lIjogdHJ1ZQp9Cg==', 'models/RSIAT_adapter.py': '...', 'scripts/build_experiments_notebook.py': '...', 'scripts/build_wave.py': 'IiIiQnVpbGQgdGhlIEthZ2dsZSBub3RlYm9vayBvZiBvbmUgd2F2ZTogbG9ja2VkIGNvZGUgYmFzZSArIGVtYmVkZGVkIHBhdGNoLgoKICBweXRob24gc2NyaXB0cy9idWlsZF93YXZlLnB5IGthZ2dsZS93YXZlcy88d2F2ZT4vd2F2ZS5qc29uCgp3YXZlLmpzb24gbmFtZXMgdGhlIGJhc2UgY29tbWl0IChwYWNrZWQgb25jZSBieSBzY3JpcHRzL3BhY2tfYmFzZS5weSkgYW5kIHRoZSBqb2JzLiBUaGUKcGF0Y2ggaXMgZXZlcnkgZmlsZSB0aGF0IGRpZmZlcnMgYmV0d2VlbiB0aGF0IGNvbW1pdCBhbmQgdGhlIHdvcmtpbmcgdHJlZSAodHJhY2tlZCBvcgpuZXcsIHNhbWUgZXhjbHVzaW9ucyBhcyB0aGUgYmFzZSwgYW5kIG5vIE1hcmtkb3duKSwgc3RvcmVkIHdpdGggdGhlIGhhc2ggb2YgaXRzIG9yaWdpbmFsCnNvIHRoZSBydW5uZXIgcmVmdXNlcyBhIHdyb25nIGJhc2UuIE91dHB1dDogPHdhdmUgZGlyPi88TkFNRT5fa2FnZ2xlLmlweW5iIHdpdGggdGhyZWUKY2VsbHMgKHdoYXQgdG8gYXR0YWNoIGFuZCBzZW5kIGJhY2s7IHJ1bi5weSA9IFBMQU4gKyBQQVRDSCArIGthZ2dsZS9ydW5uZXIucHk7IHJ1bikuCiIiIgppbXBvcnQgYmFzZTY0CmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBwcHJpbnQKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0pKQpmcm9tIHNjcmlwdHMucGFja19iYXNlIGltcG9ydCBST09ULCBnaXQsIHJlc29sdmUgICMgbm9xYTogRTQwMgoKUExBTl9LRVlTID0gKCJuYW1lIiwgImpvYnMiLCAibWF4X3BhcmFsbGVsIiwgIm92ZXJyaWRlcyIsICJ0ZXN0cyIsICJzdW1tYXJpZXMiLCAiZGF0YV9zb3VyY2VzIiwgIm1vZGVsX2ZpbGUiKQoKCmRlZiBleGNsdWRlZChwYXRoKToKICAgICIiIlNhbWUgZXhjbHVzaW9ucyBhcyB0aGUgYmFzZSAocGFja19iYXNlLkVYQ0xVREUpLCBwbHVzIE1hcmtkb3duIGFuZCB0aGUgcnVubmVyIGl0c2VsZi4iIiIKICAgIHBhcnRzID0gcGF0aC5zcGxpdCgiLyIpCiAgICByZXR1cm4gKHBhdGguZW5kc3dpdGgoKCIubWQiLCAiLmlweW5iIikpIG9yIHBhcnRzWzBdIGluICgicGFwZXIiLCAiZG9jcyIsICJkaXN0IiwgImltYWdlcyIsICJrYWdnbGUiKQogICAgICAgICAgICBvciBwYXRoLnN0YXJ0c3dpdGgoImV4cHMvYXJjaGl2ZS8iKSBvciAocGFydHNbMF0gPT0gImV4cHMiIGFuZCAicmVzdWx0cyIgaW4gcGFydHNbMjotMV0pKQoKCmRlZiBidWlsZF9wYXRjaChiYXNlKToKICAgICMgcXVvdGVwYXRoIG9mZjogbm9uLUFTQ0lJIHBhdGhzIChlLmcuIHJlc2VhcmNoX25vdGVzLykgY29tZSBiYWNrIHZlcmJhdGltLCBub3QgcXVvdGVkIGFuZCBlc2NhcGVkLgogICAgY2hhbmdlZCA9IGdpdCgiLWMiLCAiY29yZS5xdW90ZXBhdGg9b2ZmIiwgImRpZmYiLCAiLS1uYW1lLXN0YXR1cyIsICItLW5vLXJlbmFtZXMiLCBiYXNlLCAiLS0iKS5kZWNvZGUoKS5zcGxpdGxpbmVzKCkKICAgIHVudHJhY2tlZCA9IGdpdCgiLWMiLCAiY29yZS5xdW90ZXBhdGg9b2ZmIiwgImxzLWZpbGVzIiwgIi0tb3RoZXJzIiwgIi0tZXhjbHVkZS1zdGFuZGFyZCIpLmRlY29kZSgpLnNwbGl0bGluZXMoKQogICAgZmlsZXMsIGJhc2Vfc2hhLCBkZWxldGVkID0ge30sIHt9LCBbXQogICAgZW50cmllcyA9IFtsaW5lLnNwbGl0KCJcdCIsIDEpIGZvciBsaW5lIGluIGNoYW5nZWRdICsgW1siQSIsIHBdIGZvciBwIGluIHVudHJhY2tlZF0KICAgIGZvciBzdGF0dXMsIHBhdGggaW4gc29ydGVkKGVudHJpZXMsIGtleT1sYW1iZGEgZTogZVsxXSk6CiAgICAgICAgaWYgZXhjbHVkZWQocGF0aCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgc3RhdHVzID09ICJEIjoKICAgICAgICAgICAgZGVsZXRlZC5hcHBlbmQocGF0aCkKICAgICAgICAgICAgY29udGludWUKICAgICAgICBkYXRhID0gKFJPT1QgLyBwYXRoKS5yZWFkX2J5dGVzKCkKICAgICAgICBpZiBiIlwwIiBub3QgaW4gZGF0YTogICMgdGV4dDogTEYgbGlrZSB0aGUgYmxvYnMgaW4gdGhlIGJhc2UgKFdpbmRvd3MgY2hlY2tvdXRzIHVzZSBDUkxGKQogICAgICAgICAgICBkYXRhID0gZGF0YS5yZXBsYWNlKGIiXHJcbiIsIGIiXG4iKQogICAgICAgIGZpbGVzW3BhdGhdID0gYmFzZTY0LmI2NGVuY29kZShkYXRhKS5kZWNvZGUoKQogICAgICAgIGlmIHN0YXR1cyA9PSAiTSI6CiAgICAgICAgICAgIGJhc2Vfc2hhW3BhdGhdID0gaGFzaGxpYi5zaGEyNTYoZ2l0KCJzaG93IiwgInt9Ont9Ii5mb3JtYXQoYmFzZSwgcGF0aCkpKS5oZXhkaWdlc3QoKQogICAgcmV0dXJuIHsiYmFzZSI6IGJhc2UsICJidWlsdCI6IHRpbWUuc3RyZnRpbWUoIiVZLSVtLSVkICVIOiVNIiksICJmaWxlcyI6IGZpbGVzLCAiYmFzZV9zaGEiOiBiYXNlX3NoYSwKICAgICAgICAgICAgImRlbGV0ZWQiOiBkZWxldGVkfQoKCmRlZiBsaW5lcyh0ZXh0KToKICAgIHBhcnRzID0gdGV4dC5zcGxpdCgiXG4iKQogICAgcmV0dXJuIFtwICsgIlxuIiBmb3IgcCBpbiBwYXJ0c1s6LTFdXSArIFtwYXJ0c1stMV1dCgoKZGVmIGJ1aWxkKHNwZWNfcGF0aCk6CiAgICBzcGVjX3BhdGggPSBQYXRoKHNwZWNfcGF0aCkucmVzb2x2ZSgpCiAgICBzcGVjID0ganNvbi5sb2FkcyhzcGVjX3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgYmFzZSA9IHJlc29sdmUoc3BlY1siYmFzZSJdKQogICAgcGF0Y2ggPSBidWlsZF9wYXRjaChiYXNlKQogICAgcGxhbiA9IHtrOiBzcGVjW2tdIGZvciBrIGluIFBMQU5fS0VZUyBpZiBrIGluIHNwZWN9CiAgICBydW5uZXIgPSAoUk9PVCAvICJrYWdnbGUiIC8gInJ1bm5lci5weSIpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKQogICAgcnVuX3B5ID0gIiUld3JpdGVmaWxlIC9rYWdnbGUvd29ya2luZy9ydW4ucHlcbiIgKyAiXG4iLmpvaW4oWwogICAgICAgICciIiJ7fSAoYmFzZSB7fSkuIFPhu61hIFBMQU4g4bufIMSRw6J5IMSR4buDIMSR4buVaSBqb2I7IMSR4buVaSBjb2RlIHRow6wgYnVpbGQgbOG6oWkgd2F2ZS4iIiInLmZvcm1hdChzcGVjWyJuYW1lIl0sIGJhc2VbOjddKSwKICAgICAgICAiUExBTiA9ICIgKyBwcHJpbnQucGZvcm1hdChwbGFuLCB3aWR0aD0xMTAsIHNvcnRfZGljdHM9RmFsc2UpLAogICAgICAgICJQQVRDSCA9ICIgKyByZXByKHBhdGNoKSwgIiIsIHJ1bm5lcl0pCiAgICBwYXRjaGVkID0gIlxuIi5qb2luKCItIGB7fWAiLmZvcm1hdChwKSBmb3IgcCBpbiBzb3J0ZWQocGF0Y2hbImZpbGVzIl0pKSBvciAiLSAoa2jDtG5nIGPDsykiCiAgICBpbnRybyA9ICJcbiIuam9pbihbCiAgICAgICAgIiMge306IHt9Ii5mb3JtYXQoc3BlY1sibmFtZSJdLCBzcGVjWyJ0aXRsZSJdKSwgIiIsCiAgICAgICAgKnNwZWMuZ2V0KCJkZXNjcmlwdGlvbiIsIFtdKSwgIiIsCiAgICAgICAgIioqQ29kZToqKiBiYXNlIGB7fWAgKGRhdGFzZXQgYHFncnNhLWJhc2Ute31gLCB04bqhbyBi4bqxbmcgYHB5dGhvbiBzY3JpcHRzL3BhY2tfYmFzZS5weSAtLWNvbW1pdCB7fWApICIKICAgICAgICAiKyBwYXRjaCB7fSBmaWxlIG5ow7puZyB0cm9uZyBjZWxsIDIgKGJ1aWxkIHt9KToiLmZvcm1hdChiYXNlWzo3XSwgYmFzZVs6N10sIGJhc2VbOjddLCBsZW4ocGF0Y2hbImZpbGVzIl0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhdGNoWyJidWlsdCJdKSwKICAgICAgICBwYXRjaGVkLCAiIiwKICAgICAgICAiKipUcsaw4bubYyBraGkgY2jhuqF5OioqIEdQVSAqKlJUWCBQUk8gNjAwMCoqOyAqQWRkIElucHV0KiA+ICIgKyAiLCAiLmpvaW4oImB7fWAiLmZvcm1hdChpKSBmb3IgaSBpbiBzcGVjWyJpbnB1dHMiXSkKICAgICAgICArICI7IEludGVybmV0IGPDsyB0aOG7gyBPZmYuIFLhu5NpICpTYXZlIFZlcnNpb24gPiBTYXZlICYgUnVuIEFsbCAoQ29tbWl0KSouIiwgIiIsCiAgICAgICAgIioqR+G7rWkgbOG6oWk6KiogYHJlc3VsdHNfe30uemlwYCAoaG/hurdjIGBydW5zL1NVTU1BUlkudHh0YCkuIi5mb3JtYXQoc3BlY1sibmFtZSJdKV0pCiAgICBjZWxscyA9IFt7ImNlbGxfdHlwZSI6ICJtYXJrZG93biIsICJtZXRhZGF0YSI6IHt9LCAic291cmNlIjogbGluZXMoaW50cm8pfSwKICAgICAgICAgICAgIHsiY2VsbF90eXBlIjogImNvZGUiLCAibWV0YWRhdGEiOiB7fSwgImV4ZWN1dGlvbl9jb3VudCI6IE5vbmUsICJvdXRwdXRzIjogW10sICJzb3VyY2UiOiBsaW5lcyhydW5fcHkpfSwKICAgICAgICAgICAgIHsiY2VsbF90eXBlIjogImNvZGUiLCAibWV0YWRhdGEiOiB7fSwgImV4ZWN1dGlvbl9jb3VudCI6IE5vbmUsICJvdXRwdXRzIjogW10sCiAgICAgICAgICAgICAgInNvdXJjZSI6IGxpbmVzKCIhY2QgL2thZ2dsZS93b3JraW5nICYmIHB5dGhvbjMgLXUgcnVuLnB5IDI+JjEgfCB0ZWUgLWEgL2thZ2dsZS93b3JraW5nL3J1bi5sb2dcbiIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIiFybSAtcmYgL2thZ2dsZS93b3JraW5nL3JlcG8gL2thZ2dsZS93b3JraW5nLypfc3BsaXQgL2thZ2dsZS93b3JraW5nL3ZpdF9iMTZfMjI0LnB0Iil9XQogICAgbmIgPSB7ImNlbGxzIjogY2VsbHMsICJtZXRhZGF0YSI6IHsia2VybmVsc3BlYyI6IHsiZGlzcGxheV9uYW1lIjogIlB5dGhvbiAzIiwgImxhbmd1YWdlIjogInB5dGhvbiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJuYW1lIjogInB5dGhvbjMifSwgImxhbmd1YWdlX2luZm8iOiB7Im5hbWUiOiAicHl0aG9uIn19LAogICAgICAgICAgIm5iZm9ybWF0IjogNCwgIm5iZm9ybWF0X21pbm9yIjogNX0KICAgIG91dCA9IHNwZWNfcGF0aC5wYXJlbnQgLyAie31fa2FnZ2xlLmlweW5iIi5mb3JtYXQoc3BlY1sibmFtZSJdKQogICAgb3V0LndyaXRlX3RleHQoanNvbi5kdW1wcyhuYiwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MSksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICBwcmludCgie30gfCBiYXNlIHt9IHwgcGF0Y2g6IHt9e30iLmZvcm1hdChvdXQsIGJhc2VbOjddLCAiLCAiLmpvaW4oc29ydGVkKHBhdGNoWyJmaWxlcyJdKSkgb3IgIi0iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAiIHwgeG/DoTogIiArICIsICIuam9pbihwYXRjaFsiZGVsZXRlZCJdKSBpZiBwYXRjaFsiZGVsZXRlZCJdIGVsc2UgIiIpKQogICAgcmV0dXJuIG91dAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBidWlsZChzeXMuYXJndlsxXSkK', 'scripts/pack_base.py': 'IiIiTG9jayBhIGNvbW1pdCBhcyB0aGUgS2FnZ2xlIGNvZGUgYmFzZSAodXBsb2FkIG9uY2UsIHJldXNlIGZvciBldmVyeSB3YXZlKS4KCiAgcHl0aG9uIHNjcmlwdHMvcGFja19iYXNlLnB5IFstLWNvbW1pdCBIRUFEXSAgIC0+ICBkaXN0L3FncnNhLWJhc2UtPHNoYTc+LnppcAoKVGhlIHppcCBpcyBgYGdpdCBhcmNoaXZlYGAgb2YgdGhlIGNvbW1pdCAoY29tbWl0dGVkIGZpbGVzIG9ubHksIG5ldmVyIHRoZSB3b3JraW5nIHRyZWUpCndpdGhvdXQgcmVzdWx0cywgcGFwZXIgYW5kIG5vdGVib29rcywgcGx1cyBCQVNFX0NPTU1JVCBhdCBpdHMgcm9vdC4gVXBsb2FkIGl0IGFzIGEgbmV3CkthZ2dsZSBkYXRhc2V0IChvciBhIG5ldyB2ZXJzaW9uIG9mIGl0KS4gV2F2ZXMgYnVpbHQgYWdhaW5zdCB0aGlzIGNvbW1pdCB0aGVuIG9ubHkgY2FycnkKdGhlaXIgcGF0Y2ggKHNjcmlwdHMvYnVpbGRfd2F2ZS5weSkuCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHppcGZpbGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMV0KIyBOZXZlciBpbiB0aGUgYmFzZSBub3IgaW4gYSBwYXRjaDogcmVzdWx0cywgd3JpdGluZywgcGFja2FnaW5nLCBub3RlYm9va3MuCkVYQ0xVREUgPSAoImV4cHMvKiovcmVzdWx0cy8qKiIsICJleHBzL2FyY2hpdmUvKioiLCAicGFwZXIvKioiLCAiZG9jcy8qKiIsICJkaXN0LyoqIiwgImltYWdlcy8qKiIsCiAgICAgICAgICAgImthZ2dsZS93YXZlcy8qKiIsICIqLmlweW5iIikKCgpkZWYgZ2l0KCphcmdzKToKICAgIHJldHVybiBzdWJwcm9jZXNzLnJ1bihbImdpdCIsICphcmdzXSwgY3dkPVJPT1QsIGNoZWNrPVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUpLnN0ZG91dAoKCmRlZiByZXNvbHZlKGNvbW1pdCk6CiAgICByZXR1cm4gZ2l0KCJyZXYtcGFyc2UiLCBjb21taXQpLmRlY29kZSgpLnN0cmlwKCkKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29tbWl0IiwgZGVmYXVsdD0iSEVBRCIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgY29tbWl0ID0gcmVzb2x2ZShhcmdzLmNvbW1pdCkKICAgIG91dCA9IFJPT1QgLyAiZGlzdCIgLyAicWdyc2EtYmFzZS17fS56aXAiLmZvcm1hdChjb21taXRbOjddKQogICAgb3V0LnBhcmVudC5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZ2l0KCItYyIsICJjb3JlLmF1dG9jcmxmPWZhbHNlIiwgImFyY2hpdmUiLCAiLS1mb3JtYXQ9emlwIiwgIi1vIiwgc3RyKG91dCksIGNvbW1pdCwgIi0tIiwgIi4iLAogICAgICAgICooIjooZXhjbHVkZSxnbG9iKXt9Ii5mb3JtYXQocCkgZm9yIHAgaW4gRVhDTFVERSkpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShvdXQsICJhIikgYXMgejoKICAgICAgICB6LndyaXRlc3RyKCJCQVNFX0NPTU1JVCIsIGNvbW1pdCArICJcbiIpCiAgICAgICAgY291bnQgPSBsZW4oei5uYW1lbGlzdCgpKQogICAgcHJpbnQoInt9ICh7fSBmaWxlcywgezouMWZ9IE1CKTogdXBsb2FkIHRvIEthZ2dsZSBhcyB0aGUgY29kZS1iYXNlIGRhdGFzZXQiLmZvcm1hdCgKICAgICAgICBvdXQsIGNvdW50LCBvdXQuc3RhdCgpLnN0X3NpemUgLyAyICoqIDIwKSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==', 'scripts/summarize_holevo.py': '...', 'scripts/summarize_svf.py': 'IiIiU3VtbWFyaXNlIHNuYXBzaG90LXZpZXcgZnVzaW9uIChTVkYsIGV4cHMvc3ZmL1JFQURNRS5tZCkuCgpFdmVyeSBydW4gb2YgZmFtaWx5IGBgc3ZmYGAgKHJlcGxpY2F0ZSAwKSBpcyBwYXJzZWQgZnJvbSBpdHMgbG9nLiBUaGUgZnVzaW9uIHdlaWdodCBvZgpgYGZ1c2Vfc25hcF9yaWRnZWBgIGFuZCBvZiBgYGZ1c2UzX3NuYXBgYCBpcyBjaG9zZW4gb25jZSBvbiB0aGUgdHVuaW5nIHJ1biAoSW1hZ2VOZXQtUgpzZWVkIDE5OTMpIGJ5IGZpbmFsIGFjY3VyYWN5IGFuZCB0aGVuIGZyb3plbjsgdGhlIGRlY2lzaW9uIHJ1bGVzIHVzZSBoZWxkLW91dCBydW5zIG9ubHkuClJ1bGVzIGFyZSBwcmUtcmVnaXN0ZXJlZCBpbiBleHBzL3N2Zi9SRUFETUUubWQgYW5kIGZpeGVkIGJlbG93LgoKICBweXRob24gc2NyaXB0cy9zdW1tYXJpemVfc3ZmLnB5IC0tcm9vdCA8bG9ncz4gWy0tcm9vdCA8bG9ncz4gLi4uXSBbLS1vdXRwdXQgc3VtbWFyeS5tZF0gWy0tanNvbiBzdW1tYXJ5Lmpzb25dCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCBzeXMKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzeXMucGF0aC5pbnNlcnQoMCwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKZnJvbSBzY3JpcHRzLnJ1bl9sZXZlbF9zdGF0cyBpbXBvcnQgUlVOX05BTUUsIHNwbGl0X2ZhbWlseSAgIyBub3FhOiBFNDAyCmZyb20gc2NyaXB0cy5zdW1tYXJpemVfY2FfaGVhZHMgaW1wb3J0IGZtdCwgZm10X2NpLCBpbnRlcnZhbCAgIyBub3FhOiBFNDAyCmZyb20gc2NyaXB0cy5zdW1tYXJpemVfc2NmIGltcG9ydCBDTk4sIENVUlZFLCBOQU1FUywgT1JERVIsIFRBU0tTLCBjb250aW51YWwgICMgbm9xYTogRTQwMgpmcm9tIHV0aWxzLnFzZCBpbXBvcnQgRlVTRV9XRUlHSFRTLCBmdXNlM19uYW1lLCBmdXNlX25hbWUgICMgbm9xYTogRTQwMgoKIyBQcmUtcmVnaXN0ZXJlZCAoZXhwcy9zdmYvUkVBRE1FLm1kKS4gRG8gbm90IGVkaXQgYWZ0ZXIgc2VlaW5nIHJlc3VsdHMuCkZBTUlMWSA9ICJzdmYiClRVTklORyA9ICgiaW1hZ2VuZXRyIiwgMTk5MykKSEVMRF9PVVQgPSB7ImltYWdlbmV0ciI6ICgyMDE1LCAyMDI2KSwgImltYWdlbmV0YSI6ICgxOTkzLCAyMDE1LCAyMDI2KX0KTUlOX0dBSU4gPSAxLjAgICMgcnVsZSBTOiBoZWxkLW91dCBmaW5hbCBnYWluIG9mIFNWRiBvdmVyIFJTSUFULCB3aXRoIENJID4gMApTQ1JFRU5fR0FJTiA9IDAuMyAgIyBzdG9wcGluZyBydWxlIG9uIHRoZSB0dW5pbmcgcnVuOiBmdXNlMyAtIFNDRiByaWRnZSBhdCB0aGUgY2hvc2VuIHdlaWdodApTQ0ZfSEVBRCA9ICJmdXNlX3JpZGdlX3N0YXRzX2IwcDI1IgpXRUlHSFRfRkFNSUxJRVMgPSB7InNuYXAiOiBsYW1iZGEgdzogZnVzZV9uYW1lKCJzbmFwX3JpZGdlIiwgdyksICJmdXNlMyI6IGZ1c2UzX25hbWV9Ck1FQ0ggPSByZS5jb21waWxlKHIiUVNEIG1lY2ggaGVhZD0oXHcrKTogKFx7LipcfSkiKQoKCmRlZiBwYXJzZShsb2cpOgogICAgbWF0Y2ggPSBSVU5fTkFNRS5tYXRjaChsb2cuc3RlbSkKICAgIGlmIG5vdCBtYXRjaDoKICAgICAgICByZXR1cm4gTm9uZQogICAgZmFtaWx5LCByZXBsaWNhdGUgPSBzcGxpdF9mYW1pbHkobWF0Y2guZ3JvdXAoInByZWZpeCIpKQogICAgaWYgZmFtaWx5ICE9IEZBTUlMWSBvciByZXBsaWNhdGUgIT0gMDoKICAgICAgICByZXR1cm4gTm9uZQogICAgdGV4dCA9IGxvZy5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikKICAgIGN1cnZlcyA9IHtoOiBqc29uLmxvYWRzKHYpIGZvciBoLCB2IGluIENVUlZFLmZpbmRhbGwodGV4dCl9CiAgICBpZiAibGluZWFyIiBub3QgaW4gY3VydmVzOgogICAgICAgIHJldHVybiBOb25lCiAgICByb3dzLCBtZWNoID0ge30sIHt9CiAgICBmb3IgaGVhZCwgdmFsdWVzIGluIFRBU0tTLmZpbmRhbGwodGV4dCk6CiAgICAgICAgcm93cy5zZXRkZWZhdWx0KGhlYWQsIFtdKS5hcHBlbmQoanNvbi5sb2Fkcyh2YWx1ZXMpKQogICAgZm9yIGhlYWQsIHZhbHVlcyBpbiBNRUNILmZpbmRhbGwodGV4dCk6CiAgICAgICAgbWVjaFtoZWFkXSA9IGpzb24ubG9hZHModmFsdWVzKSAgIyBrZWVwcyB0aGUgbGFzdCB0YXNrCiAgICBjbm4gPSBbanNvbi5sb2Fkcyh2KSBmb3IgdiBpbiBDTk4uZmluZGFsbCh0ZXh0KV0KICAgIGhlYWRzID0ge30KICAgIGZvciBoZWFkLCBjdXJ2ZSBpbiBjdXJ2ZXMuaXRlbXMoKToKICAgICAgICBmb3JnZXR0aW5nLCBfID0gY29udGludWFsKHJvd3NbaGVhZF0pIGlmIGhlYWQgaW4gcm93cyBlbHNlIChOb25lLCBOb25lKQogICAgICAgIGxhc3QgPSByb3dzW2hlYWRdWy0xXSBpZiBoZWFkIGluIHJvd3MgZWxzZSBbXQogICAgICAgIGhlYWRzW2hlYWRdID0geyJhaWEiOiBzdGF0aXN0aWNzLm1lYW4oY3VydmUpLCAiZmluYWwiOiBjdXJ2ZVstMV0sICJmb3JnZXR0aW5nIjogZm9yZ2V0dGluZywgImN1cnZlIjogY3VydmUsCiAgICAgICAgICAgICAgICAgICAgICAgInRhc2swIjogbGFzdFswXSBpZiBsYXN0IGVsc2UgTm9uZSwgInRhc2sxIjogbGFzdFsxXSBpZiBsZW4obGFzdCkgPiAxIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICAgICAgICAicGFpcl9yYXRpbyI6IG1lY2guZ2V0KGhlYWQsIHt9KS5nZXQoInBhaXJfcmF0aW8iKSwKICAgICAgICAgICAgICAgICAgICAgICAidGFza19wcmVkaWN0aW9uIjogbWVjaC5nZXQoaGVhZCwge30pLmdldCgidGFza19wcmVkaWN0aW9uIil9CiAgICByZXR1cm4geyJkYXRhc2V0IjogbG9nLnBhcmVudC5wYXJlbnQucGFyZW50Lm5hbWUsICJzZWVkIjogaW50KG1hdGNoLmdyb3VwKCJzZWVkIikpLCAibG9nIjogc3RyKGxvZyksCiAgICAgICAgICAgICJ0YXNrcyI6IGxlbihjdXJ2ZXNbImxpbmVhciJdKSwgImNubl9jdXJ2ZSI6IGNublstMV0gaWYgY25uIGVsc2UgTm9uZSwgImhlYWRzIjogaGVhZHN9CgoKZGVmIGNvbGxlY3Qocm9vdHMpOgogICAgcnVucyA9IHt9CiAgICBmb3Igcm9vdCBpbiByb290czoKICAgICAgICBmb3IgbG9nIGluIHNvcnRlZChQYXRoKHJvb3QpLnJnbG9iKCIqLmxvZyIpKToKICAgICAgICAgICAgcnVuID0gcGFyc2UobG9nKQogICAgICAgICAgICBpZiBydW46ICAjIG9uZSBydW4gcGVyIChkYXRhc2V0LCBzZWVkKTsga2VlcCB0aGUgbG9uZ2VzdCAoY29tcGxldGVkKSBvbmUKICAgICAgICAgICAgICAgIGtleSA9IChydW5bImRhdGFzZXQiXSwgcnVuWyJzZWVkIl0pCiAgICAgICAgICAgICAgICBpZiBrZXkgbm90IGluIHJ1bnMgb3IgcnVuWyJ0YXNrcyJdID4gcnVuc1trZXldWyJ0YXNrcyJdOgogICAgICAgICAgICAgICAgICAgIHJ1bnNba2V5XSA9IHJ1bgogICAgcmV0dXJuIGxpc3QocnVucy52YWx1ZXMoKSkKCgpkZWYgY2hvb3NlX3dlaWdodHMocnVucyk6CiAgICAiIiJPbmUgd2VpZ2h0IHBlciBmdXNpb24gZmFtaWx5LCBieSBmaW5hbCBhY2N1cmFjeSBvbiB0aGUgdHVuaW5nIHJ1biAodGllczogc21hbGxlciB3ZWlnaHQpLiIiIgogICAgdHVuaW5nID0gW3IgZm9yIHIgaW4gcnVucyBpZiAoclsiZGF0YXNldCJdLCByWyJzZWVkIl0pID09IFRVTklOR10KICAgIGlmIG5vdCB0dW5pbmc6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGhlYWRzLCBjaG9zZW4gPSB0dW5pbmdbMF1bImhlYWRzIl0sIHt9CiAgICBmb3IgZmFtaWx5LCBuYW1lIGluIFdFSUdIVF9GQU1JTElFUy5pdGVtcygpOgogICAgICAgIG9wdGlvbnMgPSBbdyBmb3IgdyBpbiBGVVNFX1dFSUdIVFMgaWYgbmFtZSh3KSBpbiBoZWFkc10KICAgICAgICBpZiBvcHRpb25zOgogICAgICAgICAgICBjaG9zZW5bZmFtaWx5XSA9IG5hbWUobWF4KG9wdGlvbnMsIGtleT1sYW1iZGEgdzogKGhlYWRzW25hbWUodyldWyJmaW5hbCJdLCAtdykpKQogICAgcmV0dXJuIGNob3NlbgoKCmRlZiBkZWx0YShncm91cCwgaGVhZCwgcml2YWwsIG1ldHJpYz0iZmluYWwiKToKICAgIHJldHVybiBpbnRlcnZhbChbclsiaGVhZHMiXVtoZWFkXVttZXRyaWNdIC0gclsiaGVhZHMiXVtyaXZhbF1bbWV0cmljXSBmb3IgciBpbiBncm91cAogICAgICAgICAgICAgICAgICAgICBpZiBoZWFkIGluIHJbImhlYWRzIl0gYW5kIHJpdmFsIGluIHJbImhlYWRzIl0gYW5kIHJbImhlYWRzIl1baGVhZF1bbWV0cmljXSBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgICAgICBhbmQgclsiaGVhZHMiXVtyaXZhbF1bbWV0cmljXSBpcyBub3QgTm9uZV0pCgoKZGVmIHNjcmVlbihydW5zLCBjaG9zZW4pOgogICAgIiIiU3RvcHBpbmcgcnVsZSBvbiB0aGUgdHVuaW5nIHJ1biAoZXhwcy9zdmYvUkVBRE1FLm1kKTogb3B0aW1pc3RpYywgdXNlZCBvbmx5IHRvIGRlY2lkZSBtb3JlIHNlZWRzLiIiIgogICAgdHVuaW5nID0gW3IgZm9yIHIgaW4gcnVucyBpZiAoclsiZGF0YXNldCJdLCByWyJzZWVkIl0pID09IFRVTklOR10KICAgIGlmIG5vdCB0dW5pbmcgb3Igbm90IGNob3NlbiBvciAiZnVzZTMiIG5vdCBpbiBjaG9zZW4gb3IgU0NGX0hFQUQgbm90IGluIHR1bmluZ1swXVsiaGVhZHMiXToKICAgICAgICByZXR1cm4gTm9uZQogICAgaGVhZHMgPSB0dW5pbmdbMF1bImhlYWRzIl0KICAgIGdhcCA9IGxhbWJkYSBoZWFkOiBoZWFkc1toZWFkXVsiZmluYWwiXSAtIGhlYWRzW1NDRl9IRUFEXVsiZmluYWwiXQogICAgZ2FpbiA9IGdhcChjaG9zZW5bImZ1c2UzIl0pCiAgICByZXR1cm4geyJnYWluIjogZ2FpbiwgImdvIjogZ2FpbiA+PSBTQ1JFRU5fR0FJTiwKICAgICAgICAgICAgImJ5X3dlaWdodCI6IHtmdXNlM19uYW1lKHcpOiBnYXAoZnVzZTNfbmFtZSh3KSkgZm9yIHcgaW4gRlVTRV9XRUlHSFRTIGlmIGZ1c2UzX25hbWUodykgaW4gaGVhZHN9fQoKCmRlZiBkZWNpZGUocnVucywgY2hvc2VuKToKICAgIGhlbGQgPSBbciBmb3IgciBpbiBydW5zIGlmIHJbInNlZWQiXSBpbiBIRUxEX09VVC5nZXQoclsiZGF0YXNldCJdLCAoKSldCiAgICBleHBlY3RlZCA9IHN1bShsZW4oc2VlZHMpIGZvciBzZWVkcyBpbiBIRUxEX09VVC52YWx1ZXMoKSkKICAgIG91dCA9IHsiY2hvc2VuIjogY2hvc2VuLCAiaGVsZF9vdXQiOiBsZW4oaGVsZCksICJleHBlY3RlZCI6IGV4cGVjdGVkLCAiaG9sZHMiOiBOb25lfQogICAgaWYgbm90IGNob3NlbiBvciAic25hcCIgbm90IGluIGNob3NlbjoKICAgICAgICByZXR1cm4gb3V0CiAgICBnYWluID0gZGVsdGEoaGVsZCwgY2hvc2VuWyJzbmFwIl0sICJsaW5lYXIiKQogICAgZXh0cmEgPSBkZWx0YShoZWxkLCBjaG9zZW5bImZ1c2UzIl0sIFNDRl9IRUFEKSBpZiAiZnVzZTMiIGluIGNob3NlbiBlbHNlIGludGVydmFsKFtdKQogICAgb3V0LnVwZGF0ZShnYWluPWdhaW4sIG92ZXJfc2NmPWV4dHJhLAogICAgICAgICAgICAgICBwZXJfZGF0YXNldD17ZHM6IGRlbHRhKFtyIGZvciByIGluIGhlbGQgaWYgclsiZGF0YXNldCJdID09IGRzXSwgY2hvc2VuWyJzbmFwIl0sICJsaW5lYXIiKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGRzIGluIEhFTERfT1VUfSkKICAgIGlmIGxlbihoZWxkKSA9PSBleHBlY3RlZDoKICAgICAgICBvdXRbImhvbGRzIl0gPSAoImNpOTUiIGluIGdhaW4gYW5kIGdhaW5bIm1lYW4iXSA+PSBNSU5fR0FJTiBhbmQgZ2FpblsiY2k5NSJdWzBdID4gMAogICAgICAgICAgICAgICAgICAgICAgICBhbmQgImNpOTUiIGluIGV4dHJhIGFuZCBleHRyYVsiY2k5NSJdWzBdID4gMCkKICAgIHJldHVybiBvdXQKCgpkZWYgcmVuZGVyKHJ1bnMpOgogICAgY2hvc2VuID0gY2hvb3NlX3dlaWdodHMocnVucykKICAgIG1ldGhvZHMgPSBbKCJsaW5lYXIiLCAiUlNJQVQiKSwgKFNDRl9IRUFELCAiU0NGIHJpZGdlIiksICgic25hcF9yaWRnZSIsICJzbmFwc2hvdCBhbG9uZSIpXQogICAgaWYgY2hvc2VuOgogICAgICAgIG1ldGhvZHMgKz0gWyhjaG9zZW5bZl0sIGxhYmVsKSBmb3IgZiwgbGFiZWwgaW4gKCgic25hcCIsICJTVkYiKSwgKCJmdXNlMyIsICJTVkYgKyBTQ0YiKSkgaWYgZiBpbiBjaG9zZW5dCiAgICBsaW5lcyA9IFsiIyBTVkY6IGZ1c2lvbiB2aWV3IHBsYXN0aWMgKyBzbmFwc2hvdCB0YXNrIDAiLCAiIiwKICAgICAgICAgICAgICJUcuG7jW5nIHPhu5EgY2jhu41uIHRyw6puIHJ1biB0dW5pbmcgSW1hZ2VOZXQtUiAxOTkzOiAiICsKICAgICAgICAgICAgICgiLCAiLmpvaW4oImB7fWAiLmZvcm1hdChoKSBmb3IgaCBpbiBjaG9zZW4udmFsdWVzKCkpIGlmIGNob3NlbiBlbHNlICIqKmNoxrBhIGPDsyBydW4gdHVuaW5nKioiKSwgIiIsCiAgICAgICAgICAgICAiIyMgVOG7q25nIHJ1biIsICIiLAogICAgICAgICAgICAgInwgRGF0YXNldCB8IFNlZWQgfCBQaMawxqFuZyBwaMOhcCB8IEFJQSB8IEZpbmFsIHwgRm9yZ2V0dGluZyDihpMgfCBUYXNrIDAgfCBUYXNrIDEgfCBQYWlyIHJhdGlvIHwiLAogICAgICAgICAgICAgInwtLS18LS0tfC0tLXwtLS06fC0tLTp8LS0tOnwtLS06fC0tLTp8LS0tOnwiXQogICAgb3JkZXIgPSBsYW1iZGEgcjogKE9SREVSLmluZGV4KHJbImRhdGFzZXQiXSkgaWYgclsiZGF0YXNldCJdIGluIE9SREVSIGVsc2UgOSwgclsic2VlZCJdKQogICAgZm9yIHJ1biBpbiBzb3J0ZWQocnVucywga2V5PW9yZGVyKToKICAgICAgICB0YWcgPSAiICh0dW5pbmcpIiBpZiAocnVuWyJkYXRhc2V0Il0sIHJ1blsic2VlZCJdKSA9PSBUVU5JTkcgZWxzZSAiIgogICAgICAgIGZvciBoZWFkLCBsYWJlbCBpbiBtZXRob2RzOgogICAgICAgICAgICBpZiBoZWFkIGluIHJ1blsiaGVhZHMiXToKICAgICAgICAgICAgICAgIGggPSBydW5bImhlYWRzIl1baGVhZF0KICAgICAgICAgICAgICAgIGxpbmVzLmFwcGVuZCgifCB7fSB8IHt9e30gfCB7fSB8IHt9IHwge30gfCB7fSB8IHt9IHwge30gfCB7fSB8Ii5mb3JtYXQoCiAgICAgICAgICAgICAgICAgICAgTkFNRVMuZ2V0KHJ1blsiZGF0YXNldCJdLCBydW5bImRhdGFzZXQiXSksIHJ1blsic2VlZCJdLCB0YWcsIGxhYmVsLCBmbXQoaFsiYWlhIl0sIDIpLAogICAgICAgICAgICAgICAgICAgIGZtdChoWyJmaW5hbCJdLCAyKSwgZm10KGhbImZvcmdldHRpbmciXSwgMiksIGZtdChoWyJ0YXNrMCJdLCAxKSwgZm10KGhbInRhc2sxIl0sIDEpLAogICAgICAgICAgICAgICAgICAgIGZtdChoWyJwYWlyX3JhdGlvIl0sIDIpKSkKICAgIHMgPSBzY3JlZW4ocnVucywgY2hvc2VuKQogICAgbGluZXMgKz0gWyIiLCAiIyMgTHXhuq10IGThu6tuZyBzw6BuZyBs4buNYyAocnVuIHR1bmluZywgbOG6oWMgcXVhbikiLCAiIl0KICAgIGlmIHMgaXMgTm9uZToKICAgICAgICBsaW5lcy5hcHBlbmQoIi0gQ2jGsGEgY8OzIHJ1biB0dW5pbmcgSW1hZ2VOZXQtUiAxOTkzLiIpCiAgICBlbHNlOgogICAgICAgIGxpbmVzLmFwcGVuZCgiLSDOlCBmaW5hbCBTVkYrU0NGIOKIkiBTQ0YgcmlkZ2UgdOG6oWkgYHt9YDoge30gKG5nxrDhu6FuZyArezpnfSkg4oaSICoqe30qKiIuZm9ybWF0KAogICAgICAgICAgICBjaG9zZW5bImZ1c2UzIl0sIGZtdChzWyJnYWluIl0sIDIpLCBTQ1JFRU5fR0FJTiwKICAgICAgICAgICAgIsSRaSB0aeG6v3AgKGNo4bqheSBoZWxkLW91dCkiIGlmIHNbImdvIl0gZWxzZSAiROG7qk5HLCBraMO0bmcgY2jhuqF5IHRow6ptIHNlZWQiKSkKICAgICAgICBsaW5lcy5hcHBlbmQoIi0gVGhlbyB0cuG7jW5nIHPhu5E6ICIgKyAiLCAiLmpvaW4oInt9IHt9Ii5mb3JtYXQoaCwgZm10KHYsIDIpKSBmb3IgaCwgdiBpbiBzWyJieV93ZWlnaHQiXS5pdGVtcygpKSkKICAgIGQgPSBkZWNpZGUocnVucywgY2hvc2VuKQogICAgZFsic2NyZWVuIl0gPSBzCiAgICBsaW5lcyArPSBbIiIsICIjIyBRdXnhur90IMSR4buLbmggKHF1eSB04bqvYyDEkeG6t3QgdHLGsOG7m2MsIGNo4buJIHJ1biBoZWxkLW91dCkiLCAiIl0KICAgIGlmICJnYWluIiBub3QgaW4gZDoKICAgICAgICBsaW5lcy5hcHBlbmQoIi0gQ2jGsGEgY8OzIHJ1biB0dW5pbmc6IGNoxrBhIHF1eeG6v3QgxJHhu4tuaC4iKQogICAgZWxzZToKICAgICAgICBnLCBvID0gZFsiZ2FpbiJdLCBkWyJvdmVyX3NjZiJdCiAgICAgICAgbGluZXMuYXBwZW5kKCItIM6UIGZpbmFsIFNWRiDiiJIgUlNJQVQ6IHt9IHt9IChuPXt9KTsge30iLmZvcm1hdCgKICAgICAgICAgICAgZm10KGdbIm1lYW4iXSksIGZtdF9jaShnKSwgZ1sibiJdLCAiLCAiLmpvaW4oInt9IHt9IChuPXt9KSIuZm9ybWF0KE5BTUVTW2RzXSwgZm10KGVbIm1lYW4iXSksIGVbIm4iXSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGRzLCBlIGluIGRbInBlcl9kYXRhc2V0Il0uaXRlbXMoKSkpKQogICAgICAgIGxpbmVzLmFwcGVuZCgiLSDOlCBmaW5hbCBTVkYrU0NGIOKIkiBTQ0YgcmlkZ2U6IHt9IHt9IChuPXt9KSIuZm9ybWF0KGZtdChvWyJtZWFuIl0pLCBmbXRfY2kobyksIG9bIm4iXSkpCiAgICAgICAgdmVyZGljdCA9ICgiY2jGsGEgxJHhu6cgcnVuICh7fS97fSkiLmZvcm1hdChkWyJoZWxkX291dCJdLCBkWyJleHBlY3RlZCJdKSBpZiBkWyJob2xkcyJdIGlzIE5vbmUKICAgICAgICAgICAgICAgICAgIGVsc2UgKCLEkOG6oFQiIGlmIGRbImhvbGRzIl0gZWxzZSAiS0jDlE5HIMSQ4bqgVCIpKQogICAgICAgIGxpbmVzLmFwcGVuZCgiLSAqKlJ1bGUgUyoqIChnYWluIOKJpSB7Omd9IHbhu5tpIENJID4gMCwgdsOgIHBo4bqnbiB0aMOqbSBzbyB24bubaSBTQ0YgY8OzIENJID4gMCk6ICoqe30qKiIuZm9ybWF0KAogICAgICAgICAgICBNSU5fR0FJTiwgdmVyZGljdCkpCiAgICBjaGVja3MgPSBbInt9IHt9OiB7fSIuZm9ybWF0KE5BTUVTLmdldChyWyJkYXRhc2V0Il0sIHJbImRhdGFzZXQiXSksIHJbInNlZWQiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIj0iIGlmIHJbImhlYWRzIl1bImxpbmVhciJdWyJjdXJ2ZSJdID09IHJbImNubl9jdXJ2ZSJdIGVsc2UgIuKJoCAoTOG7lkkpIikKICAgICAgICAgICAgICBmb3IgciBpbiBzb3J0ZWQocnVucywga2V5PW9yZGVyKV0KICAgIGxpbmVzICs9IFsiIiwgIioqS2nhu4NtIHRyYSBo4buTaSBxdXkgKGhlYWQgbGluZWFyID0gQ05OIGN1cnZlKToqKiAiICsgKCI7ICIuam9pbihjaGVja3MpIG9yICJraMO0bmcgY8OzIHJ1biIpLCAiIl0KICAgIHJldHVybiAiXG4iLmpvaW4obGluZXMpICsgIlxuIiwgZAoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yb290IiwgYWN0aW9uPSJhcHBlbmQiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1qc29uIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBydW5zID0gY29sbGVjdChhcmdzLnJvb3QpCiAgICB0ZXh0LCBkZWNpc2lvbiA9IHJlbmRlcihydW5zKQogICAgaWYgYXJncy5vdXRwdXQ6CiAgICAgICAgUGF0aChhcmdzLm91dHB1dCkud3JpdGVfdGV4dCh0ZXh0LCBlbmNvZGluZz0idXRmLTgiKQogICAgaWYgYXJncy5qc29uOgogICAgICAgIHNsaW0gPSBbe2s6IHYgZm9yIGssIHYgaW4gci5pdGVtcygpIGlmIGsgIT0gImNubl9jdXJ2ZSJ9IGZvciByIGluIHJ1bnNdCiAgICAgICAgUGF0aChhcmdzLmpzb24pLndyaXRlX3RleHQoanNvbi5kdW1wcyh7InJ1bnMiOiBzbGltLCAiZGVjaXNpb24iOiBkZWNpc2lvbn0sIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KHRleHQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=', 'tests/test_holevo.py': '...', 'tests/test_svf.py': 'aW1wb3J0IGltcG9ydGxpYi51dGlsCmltcG9ydCBqc29uCmltcG9ydCB0ZW1wZmlsZQppbXBvcnQgdW5pdHRlc3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdW5pdHRlc3QgaW1wb3J0IG1vY2sKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCmZyb20gdXRpbHMgaW1wb3J0IHFzZAoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCiMgTG9hZGVkIGJ5IHBhdGg6IG9uIEthZ2dsZSBhIHNpdGUtcGFja2FnZXMgInRlc3RzIiBwYWNrYWdlIHNoYWRvd3MgdGhpcyBmb2xkZXIsIHNvCiMgImZyb20gdGVzdHMudGVzdF9xc2QgaW1wb3J0IC4uLiIgZmFpbHMgdGhlcmUuCl9zcGVjID0gaW1wb3J0bGliLnV0aWwuc3BlY19mcm9tX2ZpbGVfbG9jYXRpb24oIl9xc2RfZml4dHVyZXMiLCBQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoInRlc3RfcXNkLnB5IikpCl9maXh0dXJlcyA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoX3NwZWMpCl9zcGVjLmxvYWRlci5leGVjX21vZHVsZShfZml4dHVyZXMpCkZha2VMZWFybmVyID0gX2ZpeHR1cmVzLkZha2VMZWFybmVyClNWRl9IRUFEUyA9IFsibGluZWFyIiwgInJpZGdlX3N0YXRzIiwgInNuYXBfcmlkZ2UiXQoKCmRlZiBydW5fc2VxdWVuY2UodG1wLCB0YXNrcz0yKToKICAgIHRvcmNoLm1hbnVhbF9zZWVkKDApICAjIHRoZSBmYWtlIGhlYWRzIGFyZSBpbml0aWFsaXNlZCBmcm9tIHRoZSBnbG9iYWwgUk5HCiAgICBsZWFybmVyLCByZXN1bHRzID0gRmFrZUxlYXJuZXIoKSwgW10KICAgIGxlYXJuZXIuYXJncy51cGRhdGUocXNkX2hlYWRzPVNWRl9IRUFEUywgcXNkX2Z1c2U9VHJ1ZSwgcXNkX3NhdmVfc2NvcmVzPVRydWUpCiAgICBmb3IgdGFzayBpbiByYW5nZSh0YXNrcyk6CiAgICAgICAgbGVhcm5lci5zdGFydF90YXNrKHRhc2spCiAgICAgICAgbGluZWFyID0gbnAuYXNhcnJheShsZWFybmVyLnRlc3RfeVtsZWFybmVyLnRlc3RfeSA8IGxlYXJuZXIuX3RvdGFsX2NsYXNzZXNdKQogICAgICAgIGJlZm9yZSA9ICh0b3JjaC5nZXRfcm5nX3N0YXRlKCksIG5wLnJhbmRvbS5nZXRfc3RhdGUoKVsxXS5jb3B5KCkpCiAgICAgICAgcmVzdWx0cy5hcHBlbmQocXNkLmV2YWx1YXRlX3FzZChsZWFybmVyLCB0YXNrLCBsaW5lYXIsIHN0cihQYXRoKHRtcCkgLyAic3RhdHMiKSkpCiAgICAgICAgYXNzZXJ0IHRvcmNoLmVxdWFsKGJlZm9yZVswXSwgdG9yY2guZ2V0X3JuZ19zdGF0ZSgpKQogICAgICAgIGFzc2VydCBucC5hcnJheV9lcXVhbChiZWZvcmVbMV0sIG5wLnJhbmRvbS5nZXRfc3RhdGUoKVsxXSkKICAgIHJldHVybiBsZWFybmVyLCByZXN1bHRzCgoKY2xhc3MgU2V0dGluZ3NUZXN0KHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X2hlYWRzX2FuZF9vdmVybGF5KHNlbGYpOgogICAgICAgIGNmZyA9IHFzZC5xc2Rfc2V0dGluZ3MoeyJxc2RfaGVhZHMiOiBTVkZfSEVBRFMsICJxc2RfZnVzZSI6IFRydWV9KQogICAgICAgIGZvciB3IGluIHFzZC5GVVNFX1dFSUdIVFM6CiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4ocXNkLmZ1c2VfbmFtZSgic25hcF9yaWRnZSIsIHcpLCBjZmdbImhlYWRzIl0pCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4ocXNkLmZ1c2UzX25hbWUodyksIGNmZ1siaGVhZHMiXSkKICAgICAgICAjIFdpdGhvdXQgcmlkZ2Vfc3RhdHMgdGhlcmUgaXMgbm8gdGhyZWUtd2F5IGZ1c2lvbjsgd2l0aG91dCBxc2RfZnVzZSBubyBmdXNpb24gYXQgYWxsLgogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocXNkLnFzZF9zZXR0aW5ncyh7InFzZF9oZWFkcyI6IFsic25hcF9yaWRnZSJdLCAicXNkX2Z1c2UiOiBUcnVlfSlbImZ1c2UzIl0sIFtdKQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwocXNkLnFzZF9zZXR0aW5ncyh7InFzZF9oZWFkcyI6IFNWRl9IRUFEU30pWyJoZWFkcyJdLCBTVkZfSEVBRFMpCiAgICAgICAgIyBPbGQgb3ZlcmxheXMga2VlcCBleGFjdGx5IHRoZWlyIGhlYWRzIChTVkYgYWRkcyBub3RoaW5nIHRvIEEyIC8gU0NGIHJ1bnMpLgogICAgICAgIHNjZiA9IGpzb24ubG9hZHMoKFJPT1QgLyAiZXhwcy9zY2Yvc2NmLmpzb24iKS5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgc2VsZi5hc3NlcnRGYWxzZShhbnkoInNuYXAiIGluIGggZm9yIGggaW4gcXNkLnFzZF9zZXR0aW5ncyhzY2YpWyJoZWFkcyJdKSkKICAgICAgICBzdmYgPSBqc29uLmxvYWRzKChST09UIC8gImV4cHMvc3ZmL3N2Zi5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgICAgIGNoYW5nZWQgPSB7ayBmb3IgayBpbiBzZXQoc3ZmKSB8IHNldChzY2YpIGlmIHN2Zi5nZXQoaykgIT0gc2NmLmdldChrKX0KICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHtrIGZvciBrIGluIGNoYW5nZWQgaWYgbm90IGsuc3RhcnRzd2l0aCgicXNkIil9LCB7InByZWZpeCJ9KSAgIyBldmFsdWF0aW9uIG9ubHkKCgpjbGFzcyBEcml2ZXJUZXN0KHVuaXR0ZXN0LlRlc3RDYXNlKToKICAgIGRlZiB0ZXN0X3NuYXBzaG90X3ZpZXcoc2VsZik6CiAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXA6CiAgICAgICAgICAgIHdpdGggc2VsZi5hc3NlcnRMb2dzKGxldmVsPSJJTkZPIik6CiAgICAgICAgICAgICAgICBsZWFybmVyLCByZXN1bHRzID0gcnVuX3NlcXVlbmNlKHRtcCkKICAgICAgICAgICAgc3RhdGUgPSBsZWFybmVyLl9xc2Rfc3RhdGUKICAgICAgICAgICAgIyBUaGUgc25hcHNob3QgaXMgdGhlIG5ldHdvcmsgYWZ0ZXIgdGFzayAwLCBmcm96ZW46IGl0IGVxdWFscyB0aGUgb2xkIG5ldHdvcmsgYXQgdGFzayAxCiAgICAgICAgICAgICMgd2hpbGUgdGhlIHBsYXN0aWMgbmV0d29yayBoYXMgbW92ZWQuCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZSh0b3JjaC5lcXVhbChzdGF0ZVsic25hcHNob3QiXS53ZWlnaHQsIGxlYXJuZXIuX29sZF9uZXR3b3JrLndlaWdodCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UodG9yY2guZXF1YWwoc3RhdGVbInNuYXBzaG90Il0ud2VpZ2h0LCBsZWFybmVyLl9uZXR3b3JrLndlaWdodCkpCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoYW55KHAucmVxdWlyZXNfZ3JhZCBmb3IgcCBpbiBzdGF0ZVsic25hcHNob3QiXS5wYXJhbWV0ZXJzKCkpKQogICAgICAgICAgICAjIEF0IHRhc2sgMCB0aGUgc25hcHNob3QgdmlldyBpcyB0aGUgcGxhc3RpYyB2aWV3IGFuZCB0aGUgZmFrZSBzdG9yZWQgc3RhdGlzdGljcyBhcmUgZXhhY3QsCiAgICAgICAgICAgICMgc28gdGhlIHNuYXBzaG90IHJpZGdlIHJlcHJvZHVjZXMgcmlkZ2Vfc3RhdHMuCiAgICAgICAgICAgIG5wLnRlc3RpbmcuYXNzZXJ0X2FycmF5X2VxdWFsKHJlc3VsdHNbMF1bInByZWRpY3Rpb25zIl1bInNuYXBfcmlkZ2UiXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1swXVsicHJlZGljdGlvbnMiXVsicmlkZ2Vfc3RhdHMiXSkKICAgICAgICAgICAgIyBTdGF0aXN0aWNzIGFyZSBhcHBlbmRlZCBvbmNlIHBlciB0YXNrIGFuZCBuZXZlciByZXdyaXR0ZW4uCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoW20uc2hhcGVbMF0gZm9yIG0gaW4gc3RhdGVbInNuYXBfbWVhbnMiXV0sIFsyLCAyXSkKICAgICAgICAgICAgZmVhdHMgPSBsZWFybmVyLnRyYWluX3hbbGVhcm5lci50cmFpbl95IDwgMl0gQCBzdGF0ZVsic25hcHNob3QiXS53ZWlnaHQuZGV0YWNoKCkKICAgICAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYWxsY2xvc2Uoc3RhdGVbInNuYXBfbWVhbnMiXVswXS5udW1weSgpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBucC5zdGFjayhbZmVhdHNbbGVhcm5lci50cmFpbl95W2xlYXJuZXIudHJhaW5feSA8IDJdID09IGNdLm1lYW4oMCkubnVtcHkoKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGMgaW4gcmFuZ2UoMildKSwgYXRvbD0xZS01KQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCJzbmFwX3JpZGdlIiwgcmVzdWx0c1sxXVsiZGlhZ25vc3RpY3MiXVsiZnVzZV9zY2FsZSJdKQogICAgICAgICAgICBmb3IgdyBpbiBxc2QuRlVTRV9XRUlHSFRTOgogICAgICAgICAgICAgICAgc2VsZi5hc3NlcnRJbihxc2QuZnVzZTNfbmFtZSh3KSwgcmVzdWx0c1sxXVsiaGVhZHMiXSkKICAgICAgICAgICAgd2l0aCBucC5sb2FkKFBhdGgodG1wKSAvICJzdGF0cyIgLyAicXNkX3Njb3Jlc190YXNrXzEubnB6IikgYXMgc2F2ZWQ6CiAgICAgICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHNhdmVkWyJzbmFwX3JpZGdlIl0uc2hhcGUsICg4MCwgNCkpCiAgICAgICAgICAgICMgRGV0ZXJtaW5pc3RpYyBnaXZlbiB0aGUgc2VlZC4KICAgICAgICAgICAgXywgYWdhaW4gPSBydW5fc2VxdWVuY2UodG1wKQogICAgICAgICAgICBzZWxmLmFzc2VydEVxdWFsKHJlc3VsdHNbMV1bImhlYWRzIl0sIGFnYWluWzFdWyJoZWFkcyJdKQoKICAgIGRlZiB0ZXN0X3ZhbmlzaGluZ193ZWlnaHRfcmVjb3ZlcnNfdGhlX290aGVyX2hlYWRzKHNlbGYpOgogICAgICAgIHdpdGggbW9jay5wYXRjaC5vYmplY3QocXNkLCAiRlVTRV9XRUlHSFRTIiwgKDAuMjUsIDFlLTkpKToKICAgICAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXA6CiAgICAgICAgICAgICAgICB3aXRoIHNlbGYuYXNzZXJ0TG9ncyhsZXZlbD0iSU5GTyIpOgogICAgICAgICAgICAgICAgICAgIF8sIHJlc3VsdHMgPSBydW5fc2VxdWVuY2UodG1wKQogICAgICAgIHAgPSByZXN1bHRzWzFdWyJwcmVkaWN0aW9ucyJdCiAgICAgICAgbnAudGVzdGluZy5hc3NlcnRfYXJyYXlfZXF1YWwocFtxc2QuZnVzZTNfbmFtZSgxZS05KV0sIHBbImZ1c2VfcmlkZ2Vfc3RhdHNfYjBwMjUiXSkKICAgICAgICBucC50ZXN0aW5nLmFzc2VydF9hcnJheV9lcXVhbChwW3FzZC5mdXNlX25hbWUoInNuYXBfcmlkZ2UiLCAxZS05KV0sIHBbcXNkLmZ1c2VfbmFtZSgicmlkZ2Vfc3RhdHMiLCAxZS05KV0pCgoKY2xhc3MgU3VtbWFyeVRlc3QodW5pdHRlc3QuVGVzdENhc2UpOgogICAgZGVmIHdyaXRlX3J1bihzZWxmLCByb290LCBkYXRhc2V0LCBzZWVkLCBnYWluLCB0YXNrcz0zKToKICAgICAgICBmb2xkZXIgPSByb290IC8gImFkYXB0ZXIiIC8gZGF0YXNldCAvICIwIiAvICIyMCIKICAgICAgICBmb2xkZXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIGJhc2Vfcm93cyA9IFtbOTAuMCAtIDUgKiAodCAtIGopIGZvciBqIGluIHJhbmdlKHQgKyAxKV0gZm9yIHQgaW4gcmFuZ2UodGFza3MpXQogICAgICAgIGhlYWRzID0geyJsaW5lYXIiOiAwLjAsICJmdXNlX3JpZGdlX3N0YXRzX2IwcDI1IjogMC40LCAic25hcF9yaWRnZSI6IC0yLjB9CiAgICAgICAgZm9yIHcgaW4gcXNkLkZVU0VfV0VJR0hUUzogICMgd2VpZ2h0IDEgaXMgYmVzdCBvbiBldmVyeSBydW4KICAgICAgICAgICAgaGVhZHNbcXNkLmZ1c2VfbmFtZSgic25hcF9yaWRnZSIsIHcpXSA9IGdhaW4gLSBhYnMobnAubG9nMih3KSkKICAgICAgICAgICAgaGVhZHNbcXNkLmZ1c2UzX25hbWUodyldID0gZ2FpbiArIDAuMyAtIGFicyhucC5sb2cyKHcpKQogICAgICAgIGxpbmVzID0gW10KICAgICAgICBmb3IgdCBpbiByYW5nZSh0YXNrcyk6CiAgICAgICAgICAgIGZvciBoZWFkLCBnIGluIGhlYWRzLml0ZW1zKCk6CiAgICAgICAgICAgICAgICBsaW5lcy5hcHBlbmQoInggPT4gUVNEIHRhc2tzIGhlYWQ9e306IHt9Ii5mb3JtYXQoaGVhZCwganNvbi5kdW1wcyhbYSArIGcgZm9yIGEgaW4gYmFzZV9yb3dzW3RdXSkpKQogICAgICAgIGZvciBoZWFkLCBnIGluIGhlYWRzLml0ZW1zKCk6CiAgICAgICAgICAgIGxpbmVzLmFwcGVuZCgieCA9PiBRU0QgbWVjaCBoZWFkPXt9OiB7fSIuZm9ybWF0KGhlYWQsIGpzb24uZHVtcHMoeyJ0YXNrX3ByZWRpY3Rpb24iOiA5MC4wLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJwYWlyX3JhdGlvIjogMS4yfSkpKQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoInggPT4gUVNEIGN1cnZlIGhlYWQ9e306IHt9Ii5mb3JtYXQoaGVhZCwganNvbi5kdW1wcyhbc3VtKHIpIC8gbGVuKHIpICsgZwogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHIgaW4gYmFzZV9yb3dzXSkpKQogICAgICAgIGxpbmVzLmFwcGVuZCgieCA9PiBDTk4gdG9wMSBjdXJ2ZToge30iLmZvcm1hdChqc29uLmR1bXBzKFtzdW0ocikgLyBsZW4ocikgZm9yIHIgaW4gYmFzZV9yb3dzXSkpKQogICAgICAgIG5hbWUgPSAic3ZmX3t9X3ByZXRyYWluZWRfdml0X2IxNl8yMjRfaW4yMWtfYWRhcHRlci5sb2ciLmZvcm1hdChzZWVkKQogICAgICAgIChmb2xkZXIgLyBuYW1lKS53cml0ZV90ZXh0KCJcbiIuam9pbihsaW5lcyksIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgZGVmIHRlc3Rfd2VpZ2h0X2Zyb3plbl9vbl90dW5pbmdfcnVuX2FuZF9ydWxlKHNlbGYpOgogICAgICAgIGZyb20gc2NyaXB0cy5zdW1tYXJpemVfc3ZmIGltcG9ydCBjb2xsZWN0LCByZW5kZXIKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICAgICAgcm9vdCA9IFBhdGgodG1wKQogICAgICAgICAgICBzZWxmLndyaXRlX3J1bihyb290LCAiaW1hZ2VuZXRyIiwgMTk5MywgNS4wKSAgIyB0dW5pbmcgcnVuOiBleGNsdWRlZCBmcm9tIHRoZSBydWxlCiAgICAgICAgICAgIGZvciBpLCBzZWVkIGluIGVudW1lcmF0ZSgoMjAxNSwgMjAyNikpOgogICAgICAgICAgICAgICAgc2VsZi53cml0ZV9ydW4ocm9vdCwgImltYWdlbmV0ciIsIHNlZWQsIDEuMiArIDAuMSAqIGkpCiAgICAgICAgICAgIHRleHQsIGRlY2lzaW9uID0gcmVuZGVyKGNvbGxlY3QoW3RtcF0pKQogICAgICAgICAgICBzZWxmLmFzc2VydElzTm9uZShkZWNpc2lvblsiaG9sZHMiXSkgICMgSW1hZ2VOZXQtQSBtaXNzaW5nCiAgICAgICAgICAgIHNlbGYuYXNzZXJ0SW4oImNoxrBhIMSR4bunIHJ1biAoMi81KSIsIHRleHQpCiAgICAgICAgICAgICMgVHVuaW5nIHJ1bjogZnVzZTMgYXQgd2VpZ2h0IDEgaXMgNS4zIGFnYWluc3QgU0NGIDAuNCAtPiBnbzsgd2VpZ2h0IDQgaXMgMiBiZWxvdyBpdHMgcGVhay4KICAgICAgICAgICAgc2VsZi5hc3NlcnRUcnVlKGRlY2lzaW9uWyJzY3JlZW4iXVsiZ28iXSkKICAgICAgICAgICAgc2VsZi5hc3NlcnRBbG1vc3RFcXVhbChkZWNpc2lvblsic2NyZWVuIl1bImdhaW4iXSwgNC45KQogICAgICAgICAgICBzZWxmLmFzc2VydEFsbW9zdEVxdWFsKGRlY2lzaW9uWyJzY3JlZW4iXVsiYnlfd2VpZ2h0Il1bImZ1c2UzX3NuYXBfYjQiXSwgMi45KQogICAgICAgICAgICBzZWxmLmFzc2VydEluKCLEkWkgdGnhur9wIiwgdGV4dCkKICAgICAgICAgICAgZm9yIGksIHNlZWQgaW4gZW51bWVyYXRlKCgxOTkzLCAyMDE1LCAyMDI2KSk6CiAgICAgICAgICAgICAgICBzZWxmLndyaXRlX3J1bihyb290LCAiaW1hZ2VuZXRhIiwgc2VlZCwgMS4xICsgMC4yICogaSkKICAgICAgICAgICAgdGV4dCwgZGVjaXNpb24gPSByZW5kZXIoY29sbGVjdChbdG1wXSkpCiAgICAgICAgc2VsZi5hc3NlcnRFcXVhbChkZWNpc2lvblsiY2hvc2VuIl0sIHsic25hcCI6ICJmdXNlX3NuYXBfcmlkZ2VfYjEiLCAiZnVzZTMiOiAiZnVzZTNfc25hcF9iMSJ9KQogICAgICAgIHNlbGYuYXNzZXJ0RXF1YWwoZGVjaXNpb25bImdhaW4iXVsibiJdLCA1KQogICAgICAgIHNlbGYuYXNzZXJ0VHJ1ZShkZWNpc2lvblsiaG9sZHMiXSkKICAgICAgICBzZWxmLmFzc2VydEluKCLEkOG6oFQiLCB0ZXh0KQogICAgICAgIHNlbGYuYXNzZXJ0Tm90SW4oIuKJoCIsIHRleHQpCgogICAgZGVmIHRlc3Rfc3RvcHBpbmdfcnVsZV9vbl90dW5pbmdfcnVuKHNlbGYpOgogICAgICAgIGZyb20gc2NyaXB0cy5zdW1tYXJpemVfc3ZmIGltcG9ydCBjb2xsZWN0LCByZW5kZXIKICAgICAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICAgICAgc2VsZi53cml0ZV9ydW4oUGF0aCh0bXApLCAiaW1hZ2VuZXRyIiwgMTk5MywgMC4zKSAgIyBmdXNlMyBhdCB3PTE6IDAuNiwgU0NGIHJpZGdlIDAuNCAtPiArMC4yIDwgMC4zCiAgICAgICAgICAgIHRleHQsIGRlY2lzaW9uID0gcmVuZGVyKGNvbGxlY3QoW3RtcF0pKQogICAgICAgIHNlbGYuYXNzZXJ0RmFsc2UoZGVjaXNpb25bInNjcmVlbiJdWyJnbyJdKQogICAgICAgIHNlbGYuYXNzZXJ0SW4oIkThu6pORyIsIHRleHQpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHVuaXR0ZXN0Lm1haW4oKQo=', 'utils/holevo.py': '...', 'utils/qsd.py': '...'}, 'base_sha': {'.gitignore': 'ca7d03b451aaced01d32656f58ce7f5dbf6352b4503dca225088f97fe19926df', 'models/RSIAT_adapter.py': '402c501c7ce1d551eca3b7f07d85aeddd01ceeccee1072a5efb5c66cd11e6f5a', 'scripts/build_experiments_notebook.py': '0ab70fcfad36e09e746cf2c96eedb488e702dfc13709cb018b323270d8f95207', 'utils/qsd.py': '7bd64fc44cb7c75242d46fc36bbcdaaeabafeb8fce437d3f94d609bc03f1e734'}, 'deleted': []}

"""Kaggle runner for one wave (template).

scripts/build_wave.py writes run.py = ``PLAN`` + ``PATCH`` + this file. The code base is
the locked zip made by scripts/pack_base.py (uploaded once as a Kaggle dataset); this
runner checks its commit, applies the wave's patch (files changed since that commit),
then runs every job and packs the logs. Datasets and the ViT weights are attached inputs.

Edit PLAN in the notebook cell to change jobs; anything else needs a rebuild.
"""
import base64
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import threading
import time
import zipfile
from pathlib import Path

INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path.cwd() / "input"
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd() / "work"
WORK.mkdir(parents=True, exist_ok=True)
REPO, OUT = WORK / "repo", WORK / "runs"
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TIMM_FUSED_ATTN="0")
N_CLASSES = {"imageneta": 200, "imagenetr": 200, "cifar224": 100, "cub": 200}
NAMES = {"imageneta": r"imagenet[-_]?a\b|imagenet-a|adversarial", "imagenetr": r"imagenet[-_]?r\b|imagenet-r|rendition"}
IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SHOW = re.compile(r"CNN top1 curve|Average Accuracy|Error|Traceback|WARNING")
JOBS = [(ds, v, s) for datasets, seeds, variants in PLAN["jobs"] for ds in datasets for s in seeds for v in variants]


def say(*args):
    print(*args, flush=True)


def walk(root, dirs=False):
    """Files (or dirs) under root, following symlinks (Kaggle mounts some inputs as symlinks)."""
    seen = set()
    for top, dnames, fnames in os.walk(root, followlinks=True):
        real = os.path.realpath(top)
        if real in seen:
            dnames[:] = []
            continue
        seen.add(real)
        for name in (dnames if dirs else fnames):
            yield Path(top) / name


def sha256(data):
    return hashlib.sha256(data).hexdigest()


# ----------------------------------------------------------------------------- environment
def check_gpu():
    probe = ("import torch, json; ok = torch.cuda.is_available(); cap = torch.cuda.get_device_capability(0) if ok else None;"
             "print(json.dumps({'torch': torch.__version__, 'ok': ok, 'name': torch.cuda.get_device_name(0) if ok else None,"
             "'cap': cap, 'arch': torch.cuda.get_arch_list()}))")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    assert r.returncode == 0, "Không import được torch:\n" + r.stderr[-2000:]
    info = json.loads(r.stdout.strip().splitlines()[-1])
    say("GPU:", info["name"], "| torch", info["torch"])
    assert info["ok"], "Không thấy GPU CUDA: bật accelerator"
    need = "sm_%d%d" % tuple(info["cap"])
    wheels = sorted({p.parent for p in walk(INPUT) if p.suffix == ".whl"})
    missing = [m for m in ("timm", "tqdm", "scipy") if subprocess.run([sys.executable, "-c", "import " + m]).returncode]
    if need not in info["arch"] or missing:
        assert wheels, "Thiếu {} và không có wheel offline trong input".format(missing or ["torch cho " + need])
        cmd = [sys.executable, "-m", "pip", "install", "--no-index"] + sum((["--find-links", str(d)] for d in wheels), [])
        subprocess.run(cmd + (["torch", "torchvision"] if need not in info["arch"] else []) + missing, check=True)
        say("Đã cài package từ wheel → chạy lại run.py trong tiến trình mới")
        os.execv(sys.executable, [sys.executable] + sys.argv)


# ----------------------------------------------------------------------------- code: locked base + patch
def find_base():
    """Folder (or zip) of the locked base: it holds BASE_COMMIT at its root."""
    for p in walk(INPUT):
        if p.name == "BASE_COMMIT" and (p.parent / "main.py").is_file():
            return p.parent
    for p in walk(INPUT):
        if p.suffix == ".zip" and zipfile.is_zipfile(p):
            with zipfile.ZipFile(p) as z:
                if "BASE_COMMIT" in z.namelist():
                    target = WORK / "base_unzipped"
                    shutil.rmtree(target, ignore_errors=True)
                    z.extractall(target)
                    return target
    raise SystemExit("STOP: không thấy code base (zip/folder có BASE_COMMIT). Add Input dataset từ dist/qgrsa-base-*.zip")


def prepare_code():
    base = find_base()
    commit = (base / "BASE_COMMIT").read_text().strip()
    if commit != PATCH["base"]:
        raise SystemExit("STOP: code base là commit {} nhưng wave cần {} → attach đúng version dataset base".format(
            commit[:7], PATCH["base"][:7]))
    shutil.rmtree(REPO, ignore_errors=True)
    shutil.copytree(base, REPO, copy_function=shutil.copyfile, ignore=shutil.ignore_patterns("__pycache__"))
    for path, expected in PATCH["base_sha"].items():  # the patch is made for exactly these originals
        target = REPO / path
        actual = sha256(target.read_bytes()) if target.is_file() else None
        if actual != expected:
            raise SystemExit("STOP: {} trong base khác bản patch được build từ đó".format(path))
    for path, blob in PATCH["files"].items():
        target = REPO / path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(blob))
    for path in PATCH["deleted"]:
        (REPO / path).unlink(missing_ok=True)
    say("Code: base {} + patch {} file, xoá {} (build {})".format(commit[:7], len(PATCH["files"]), len(PATCH["deleted"]),
                                                               PATCH["built"]))
    os.chdir(REPO)
    sys.path.insert(0, str(REPO))


# ----------------------------------------------------------------------------- model and data
def prepare_model():
    import torch
    cands = [p for p in walk(INPUT) if p.suffix.lower() in {".pt", ".pth", ".bin", ".safetensors", ".npz"}
             and p.stat().st_size > 50 * 2 ** 20]
    if PLAN.get("model_file"):
        cands = [Path(PLAN["model_file"])]
    assert len(cands) == 1, "Cần đúng 1 file ViT-B/16 in21k trong input, thấy {} → điền PLAN['model_file']".format(cands)
    src = cands[0]
    if src.suffix == ".safetensors":
        from safetensors.torch import load_file
        state = load_file(str(src))
    elif src.suffix == ".npz":
        from timm.models.vision_transformer import VisionTransformer, _load_weights
        vit = VisionTransformer(img_size=224, patch_size=16, embed_dim=768, depth=12, num_heads=12, num_classes=0)
        _load_weights(vit, str(src))
        state = vit.state_dict()
    else:
        state = torch.load(src, map_location="cpu", weights_only=False)
        for k in ("state_dict", "model", "model_state_dict"):
            if isinstance(state, dict) and isinstance(state.get(k), dict):
                state = state[k]
                break
    state = {k.removeprefix("module.").removeprefix("model.").removeprefix("vit."): v
             for k, v in state.items() if torch.is_tensor(v)}
    if "embeddings.patch_embeddings.projection.weight" in state:  # Hugging Face -> timm
        hf, state = state, {}
        for a, b in {"embeddings.cls_token": "cls_token", "embeddings.position_embeddings": "pos_embed",
                     "embeddings.patch_embeddings.projection.weight": "patch_embed.proj.weight",
                     "embeddings.patch_embeddings.projection.bias": "patch_embed.proj.bias",
                     "layernorm.weight": "norm.weight", "layernorm.bias": "norm.bias"}.items():
            state[b] = hf[a]
        ren = {"layernorm_before": "norm1", "layernorm_after": "norm2", "attention.output.dense": "attn.proj",
               "intermediate.dense": "mlp.fc1", "output.dense": "mlp.fc2"}
        for i in range(12):
            b = "encoder.layer.{}.".format(i)
            for s in ("weight", "bias"):
                for o, n in ren.items():
                    state["blocks.{}.{}.{}".format(i, n, s)] = hf[b + o + "." + s]
                state["blocks.{}.attn.qkv.{}".format(i, s)] = torch.cat(
                    [hf[b + "attention.attention.{}.{}".format(q, s)] for q in ("query", "key", "value")])
    state = {k: v for k, v in state.items() if not k.startswith(("head", "pre_logits", "classifier", "pooler"))}
    assert tuple(state["pos_embed"].shape) == (1, 197, 768), "Cần ViT-B/16 224 (định dạng timm, npz hoặc safetensors)"
    out = WORK / "vit_b16_224.pt"
    torch.save(state, out)
    say("Model:", src)
    return out


def extract(archive, dest):
    dest.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
    else:
        with tarfile.open(archive) as t:
            t.extractall(dest, filter="data")
    return dest


def raw_200(dataset):
    pattern = re.compile(NAMES[dataset], re.I)

    def search(root):
        return [p for p in walk(root, dirs=True) if WORK not in p.parents and pattern.search(str(p))
                and sum(c.is_dir() for c in p.iterdir()) == 200]
    found = search(INPUT)
    if not found:
        archives = [p for p in walk(INPUT) if pattern.search(p.name) and (zipfile.is_zipfile(p) or tarfile.is_tarfile(p))]
        assert archives, "Không thấy {} (folder 200 lớp hoặc archive) → điền PLAN['data_sources']".format(dataset)
        found = [p for p in walk(extract(archives[0], WORK / (dataset + "_raw")), dirs=True)
                 if sum(c.is_dir() for c in p.iterdir()) == 200]
    presplit = {p.parent for p in found if p.name == "train" and p.parent / "test" in found}
    if len(presplit) == 1 and len(found) == 2:
        return presplit.pop()  # e.g. Kaggle zaphat206/imagenet-r
    assert len(found) == 1, "{}: cần đúng 1 folder 200 lớp, thấy {}".format(dataset, found)
    return found[0]


def split_80_20(raw, dataset):
    """ImageNet-A protocol of every round: 80/20 per class, seed 67 (symlinks)."""
    import numpy as np
    root = WORK / (dataset + "_split")
    if not (root / "test").is_dir():
        rng = np.random.default_rng(67)
        for cls in sorted(c for c in raw.iterdir() if c.is_dir()):
            files = sorted(f for f in cls.iterdir() if f.suffix.lower() in IMG)
            order = rng.permutation(len(files)).tolist()
            n_tr = min(max(1, round(len(files) * 0.8)), len(files) - 1)
            for split, idx in (("train", order[:n_tr]), ("test", order[n_tr:])):
                d = root / split / cls.name
                d.mkdir(parents=True, exist_ok=True)
                for i in idx:
                    (d / files[i].name).symlink_to(files[i].resolve())
    return root


def cifar_root():
    found = [p.parent for p in walk(INPUT, dirs=True) if p.name == "cifar-100-python" and (p / "train").is_file()]
    if not found:
        archives = [p for p in walk(INPUT) if "cifar" in p.name.lower() and (zipfile.is_zipfile(p) or tarfile.is_tarfile(p))]
        assert archives, "Không thấy cifar-100-python"
        found = [p.parent for p in walk(extract(archives[0], WORK / "cifar_raw"), dirs=True)
                 if p.name == "cifar-100-python" and (p / "train").is_file()]
    return found[0]


def cub_root():
    raw = [p for p in walk(INPUT, dirs=True) if (p / "train_test_split.txt").is_file() and (p / "images").is_dir()]
    if raw:
        from data.data import prepare_cub_split
        say("CUB split chính thức:", prepare_cub_split(str(raw[0]), str(WORK / "cub_split")))
        return WORK / "cub_split"
    pre = [p for p in walk(INPUT, dirs=True) if re.search(r"cub|bird", str(p), re.I) and (p / "train").is_dir()
           and (p / "test").is_dir()]
    assert len(pre) == 1, "CUB: cần bản gốc có train_test_split.txt hoặc đúng 1 bản train/+test/, thấy {}".format(pre)
    return pre[0]


def prepare_data():
    roots = {}
    for ds in sorted({ds for ds, _, _ in JOBS}):
        given = PLAN.get("data_sources", {}).get(ds)
        if given:
            src = Path(given)
        elif ds == "cifar224":
            src = cifar_root()
        elif ds == "cub":
            src = cub_root()
        else:
            src = raw_200(ds)
        roots[ds] = src if ds in ("cifar224", "cub") or (src / "test").is_dir() else split_80_20(src, ds)
        say("Data", ds, "→", roots[ds])
    return roots


# ----------------------------------------------------------------------------- jobs
def run_tests():
    for pattern in PLAN.get("tests", []):
        r = subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-p", pattern],
                           capture_output=True, text=True)
        say("test", pattern, r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "")
        if r.returncode:
            raise SystemExit("STOP: test lỗi\n" + r.stderr[-3000:])


def run_config(ds, variant):
    from utils.qbridge_experiment import merge_dict
    found = list(REPO.glob("exps/*/{}.json".format(variant)))
    assert len(found) == 1, "Overlay {}: thấy {}".format(variant, found)
    base = json.loads((REPO / "exps/rsiat" / "{}.json".format(ds)).read_text())
    return merge_dict(merge_dict(base, json.loads(found[0].read_text())), PLAN.get("overrides", {}))


def log_path(ds, variant, seed):
    cfg = run_config(ds, variant)
    init = 0 if cfg["init_cls"] == cfg["increment"] else cfg["init_cls"]
    return OUT / "logs/adapter" / ds / str(init) / str(cfg["increment"]) / "{}_{}_{}.log".format(
        cfg["prefix"], seed, cfg["convnet_type"])


def done(ds, variant, seed):
    cfg, p = run_config(ds, variant), log_path(ds, variant, seed)
    tasks = 1 + (N_CLASSES[ds] - cfg["init_cls"]) // cfg["increment"]
    return p.is_file() and any(l.count(",") == tasks - 1 for l in p.read_text().splitlines() if "CNN top1 curve" in l)


def start(job, roots, pretrained, threads):
    ds, variant, seed = job
    log_path(*job).unlink(missing_ok=True)
    cfg = run_config(ds, variant)
    cfg.update(seed=[seed], device=[0], data_root=str(roots[ds]), pretrained_path=str(pretrained),
               output_root=str(OUT), resume=False)
    name = "{}_{}_{}".format(*job)
    path = OUT / "configs" / (name + ".json")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(cfg, indent=2))
    (OUT / "console").mkdir(exist_ok=True)
    env = dict(os.environ, OMP_NUM_THREADS=str(threads), MKL_NUM_THREADS=str(threads), PYTHONUNBUFFERED="1")
    proc = subprocess.Popen([sys.executable, "-u", "-B", "main.py", "--config", str(path)], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def pump():
        with open(OUT / "console" / (name + ".txt"), "w") as log:
            for line in proc.stdout:
                log.write(line)
                if SHOW.search(line):
                    say("[{}]".format(name), line.rstrip()[-200:])
    thread = threading.Thread(target=pump, daemon=True)
    thread.start()
    say("START [{}]".format(name))
    return proc, thread


def run_all(roots, pretrained):
    queue = [j for j in JOBS if not done(*j)]
    parallel = int(PLAN.get("max_parallel", 3))
    threads = max(1, (os.cpu_count() or 1) // parallel // 2)
    running, failed, t0 = {}, [], time.time()
    say("Jobs: {} (bỏ qua {} đã xong), {} song song".format(len(queue), len(JOBS) - len(queue), parallel))
    while queue or running:
        while queue and len(running) < parallel:
            job = queue.pop(0)
            running[job] = start(job, roots, pretrained, threads)
            time.sleep(20)  # stagger start-up
        for job, (proc, thread) in list(running.items()):
            if proc.poll() is not None:
                thread.join()
                ok = proc.returncode == 0 and done(*job)
                say(("DONE " if ok else "FAILED ") + "[{}_{}_{}]".format(*job) + " sau {:.0f} phút".format((time.time() - t0) / 60))
                if not ok:
                    failed.append(job)
                del running[job]
        time.sleep(10)
    return failed


def summarize(failed):
    lines = ["===== SUMMARY {} | {} =====".format(PLAN["name"], time.strftime("%Y-%m-%d %H:%M"))]
    for job in JOBS:
        p = log_path(*job)
        last = [l.split("=> ")[-1] for l in p.read_text().splitlines() if "Average Accuracy (CNN)" in l][-1:] if p.is_file() else []
        lines.append("{}_{}_{}: {}".format(*job, last[0] if last else "KHÔNG CÓ KẾT QUẢ"))
    if failed:
        lines.append("FAILED: {} → xem runs/console/".format(failed))
    for script in PLAN.get("summaries", []):
        out = OUT / (Path(script).stem + ".md")
        r = subprocess.run([sys.executable, "-B", script, "--root", str(OUT / "logs"), "--output", str(out),
                            "--json", str(out.with_suffix(".json"))], capture_output=True, text=True)
        lines.append(out.read_text(encoding="utf-8") if r.returncode == 0 else "{} lỗi:\n{}".format(script, r.stderr[-2000:]))
    text = "\n".join(lines) + "\n"
    (OUT / "SUMMARY.txt").write_text(text, encoding="utf-8")
    say(text)


def package():
    import torch
    import timm
    pkg = WORK / "package"
    shutil.rmtree(pkg, ignore_errors=True)
    shutil.copytree(OUT, pkg, ignore=shutil.ignore_patterns("*.pkl", "*.pt", "ckpt"))
    (pkg / "environment.json").write_text(json.dumps({
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "cuda": torch.version.cuda,
        "timm": timm.__version__, "plan": PLAN, "jobs": JOBS, "base": PATCH["base"], "patch_built": PATCH["built"],
        "patch_files": sorted(PATCH["files"])}, indent=2))
    say("Kết quả:", shutil.make_archive(str(WORK / "results_{}".format(PLAN["name"])), "zip", pkg))


def main():
    check_gpu()
    prepare_code()
    run_tests()
    roots, pretrained = prepare_data(), prepare_model()
    failed = run_all(roots, pretrained)
    summarize(failed)
    package()
    shutil.rmtree(WORK / "base_unzipped", ignore_errors=True)


if __name__ == "__main__":
    main()


In [ ]:
!cd /kaggle/working && python3 -u run.py 2>&1 | tee -a /kaggle/working/run.log
!rm -rf /kaggle/working/repo /kaggle/working/*_split /kaggle/working/vit_b16_224.pt